# LeetCode 381 — Insert Delete GetRandom O(1) - Duplicates allowed bằng Python (Hard)

Notebook này giảng nhập môn một bài **thiết kế class** (design): tự xây một "cái túi" chứa số, cho phép **số trùng nhau**, mà cả ba thao tác thêm / xóa / lấy ngẫu nhiên đều nhanh `O(1)` trung bình.

```python
class RandomizedCollection:

    def __init__(self):


    def insert(self, val: int) -> bool:


    def remove(self, val: int) -> bool:


    def getRandom(self) -> int:



# Your RandomizedCollection object will be instantiated and called as such:
# obj = RandomizedCollection()
# param_1 = obj.insert(val)
# param_2 = obj.remove(val)
# param_3 = obj.getRandom()
```

Mục tiêu của bài học:

1. Hiểu **multiset** (tập hợp cho phép phần tử trùng) và yêu cầu "xác suất **tỉ lệ thuận với số lần xuất hiện**": với `[1, 1, 2]` thì `1` ra với xác suất `2/3`, `2` ra với xác suất `1/3`.
2. Nắm chắc **chi phí thao tác** của `list`, `dict`, `set` trong Python: cái gì `O(1)`, cái gì `O(n)`, và **tại sao**.
3. **Ôn lại bài 380** (Insert Delete GetRandom O(1), không có trùng): `list` + `dict` giá trị → chỉ số, mẹo **"đổi chỗ với phần tử cuối rồi `pop`"**.
4. Đi từ cách người mới tự nghĩ ra (chỉ dùng `list`; dùng `dict` đếm) → hiểu mỗi cách vướng ở đâu → cách chính: **`list` + `dict` giá trị → `set` các chỉ số**.
5. Trace thật kỹ 3 tình huống của `remove`: trường hợp bình thường, **phần tử cần xóa chính là phần tử cuối**, và **giá trị bị xóa trùng với giá trị ở cuối**; hiểu vì sao **thứ tự 2 dòng `add` / `discard`** quyết định đúng sai.
6. Giải thích được **vì sao dùng `set` (không phải `list`) để lưu chỉ số**, và biết một cách nâng cao dùng `list` kèm "con trỏ ngược".
7. Tự **kiểm chứng phân phối** của `getRandom` bằng thực nghiệm (`Counter` trên hàng chục nghìn lần gọi), test chéo ngẫu nhiên với `random.seed`, và tránh các lỗi kinh điển.

## 1. Đề bài

Đề LeetCode (dịch sát nghĩa):

> `RandomizedCollection` là một cấu trúc dữ liệu chứa một tập các số, **có thể có số trùng** (tức là một **multiset**). Nó cần hỗ trợ thêm và xóa các phần tử cụ thể, và trả về một phần tử ngẫu nhiên.
>
> Hãy cài đặt class `RandomizedCollection`:
>
> - `RandomizedCollection()` khởi tạo object `RandomizedCollection` rỗng.
> - `bool insert(int val)` thêm phần tử `val` vào multiset, **kể cả khi phần tử đó đã có**. Trả về `true` nếu phần tử **chưa có** trước đó, `false` nếu ngược lại.
> - `bool remove(int val)` xóa phần tử `val` khỏi multiset nếu có. Trả về `true` nếu phần tử **có mặt**, `false` nếu ngược lại. Lưu ý: nếu `val` xuất hiện nhiều lần trong multiset, ta **chỉ xóa một lần xuất hiện**.
> - `int getRandom()` trả về một phần tử ngẫu nhiên trong multiset hiện tại. Xác suất mỗi phần tử được trả về **tỉ lệ tuyến tính với số lần giá trị đó xuất hiện** trong multiset.
>
> Bạn phải cài đặt các hàm sao cho **mỗi hàm chạy trung bình `O(1)`**.
>
> **Lưu ý:** Test case được sinh sao cho `getRandom` chỉ được gọi khi `RandomizedCollection` có ít nhất một phần tử.

**Example 1**

```text
Input
["RandomizedCollection", "insert", "insert", "insert", "getRandom", "remove", "getRandom"]
[[], [1], [1], [2], [], [1], []]
Output
[null, true, false, true, 2, true, 1]

Explanation
RandomizedCollection randomizedCollection = new RandomizedCollection();
randomizedCollection.insert(1);   // trả về true vì collection chưa chứa 1. Thêm 1 vào.
randomizedCollection.insert(1);   // trả về false vì collection đã chứa 1. Thêm một số 1 nữa. Collection: [1,1].
randomizedCollection.insert(2);   // trả về true vì collection chưa chứa 2. Thêm 2 vào. Collection: [1,1,2].
randomizedCollection.getRandom(); // getRandom nên:
                                  // - trả về 1 với xác suất 2/3, hoặc
                                  // - trả về 2 với xác suất 1/3.
randomizedCollection.remove(1);   // trả về true vì collection chứa 1. Xóa một số 1. Collection: [1,2].
randomizedCollection.getRandom(); // getRandom trả về 1 hoặc 2, khả năng như nhau.
```

**Constraints**

- `-2^31 <= val <= 2^31 - 1`
- Có tối đa `2 * 10^5` lời gọi `insert`, `remove` và `getRandom` (tính chung).
- Luôn có ít nhất một phần tử trong cấu trúc dữ liệu khi `getRandom` được gọi.

### Ký hiệu dùng suốt notebook

- `n` = số phần tử **đang có** trong collection (đếm cả số trùng). Với `[1, 1, 2]` thì `n = 3`.
- `k` = số **giá trị khác nhau** đang có. Với `[1, 1, 2]` thì `k = 2`.
- `c` = số lần xuất hiện của một giá trị cụ thể. Với `[1, 1, 2]`, giá trị `1` có `c = 2`.
- `q` = tổng số lời gọi (theo đề `q <= 2·10^5`).

### 1.1 Đọc input / output của bài "thiết kế class"

LeetCode mô tả một **kịch bản gọi hàm** bằng 2 list song song: list thứ nhất là **tên lệnh**, list thứ hai là **tham số** của lệnh tương ứng.

| Vị trí | Tên lệnh | Tham số | Nghĩa là | Collection sau lệnh | Output |
|---:|---|---|---|---|---|
| 0 | `"RandomizedCollection"` | `[]` | `obj = RandomizedCollection()` | `[]` | `null` |
| 1 | `"insert"` | `[1]` | `obj.insert(1)` | `[1]` | `true` (trước đó chưa có 1) |
| 2 | `"insert"` | `[1]` | `obj.insert(1)` | `[1, 1]` | `false` (đã có 1 rồi — nhưng **vẫn thêm**) |
| 3 | `"insert"` | `[2]` | `obj.insert(2)` | `[1, 1, 2]` | `true` |
| 4 | `"getRandom"` | `[]` | `obj.getRandom()` | `[1, 1, 2]` | `1` hoặc `2` (ở đây ra `2`) |
| 5 | `"remove"` | `[1]` | `obj.remove(1)` | `[1, 2]` | `true` (chỉ xóa **một** số 1) |
| 6 | `"getRandom"` | `[]` | `obj.getRandom()` | `[1, 2]` | `1` hoặc `2` (ở đây ra `1`) |

Ba chi tiết người mới hay đọc nhầm:

- `insert` trả `false` **không có nghĩa là không thêm**. Nó vẫn thêm; giá trị trả về chỉ cho biết "trước lúc thêm, `val` đã có mặt chưa".
- `remove` chỉ xóa **một** lần xuất hiện, không xóa hết.
- Output của `getRandom` là **ngẫu nhiên**, nên LeetCode không so khớp từng con số; nó kiểm tra phân phối (tần suất) qua rất nhiều lần gọi. Vì vậy khi tự test, ta sẽ kiểm tra "kết quả có **thuộc** collection lúc đó không" và đo tần suất bằng thực nghiệm.

### Vì sao bài này là Hard?

Từng thao tác riêng lẻ đều dễ làm nhanh với **một** cấu trúc dữ liệu phù hợp:

- `getRandom` nhanh cần các phần tử nằm trong một **mảng** (`list`) để chọn ngẫu nhiên **một chỉ số**.
- `remove(val)` nhanh cần biết **ngay** `val` nằm ở đâu, tức là cần tra cứu kiểu **hash** (`dict` / `set`).

Cái khó là **cả ba** phải cùng nhanh, và lại có **số trùng**: một giá trị có thể nằm ở nhiều chỗ trong mảng. Bài 380 (không trùng) đã là Medium; thêm số trùng làm phần `remove` có thêm hai tình huống "bẫy" mà ta sẽ mổ xẻ kỹ ở mục 8.

In [1]:
from fractions import Fraction
from collections import Counter

bag = [1, 1, 2]                      # collection sau 3 lệnh insert của Example 1
cnt = Counter(bag)                   # đếm số lần xuất hiện của từng giá trị
n = len(bag)
for val, c in cnt.items():
    print(f"giá trị {val}: xuất hiện {c} lần -> xác suất = {c}/{n} = {Fraction(c, n)} ≈ {c / n:.3f}")

bag2 = [1, 2]                        # sau remove(1)
for val, c in Counter(bag2).items():
    print(f"sau remove(1): giá trị {val} -> xác suất {Fraction(c, len(bag2))}")

giá trị 1: xuất hiện 2 lần -> xác suất = 2/3 = 2/3 ≈ 0.667
giá trị 2: xuất hiện 1 lần -> xác suất = 1/3 = 1/3 ≈ 0.333
sau remove(1): giá trị 1 -> xác suất 1/2
sau remove(1): giá trị 2 -> xác suất 1/2


## 2. Nền tảng

### 2.1 Multiset là gì? So sánh `list`, `set`, `Counter`

- **Set** (tập hợp) trong toán: mỗi phần tử có **hoặc không**, không có chuyện "có 2 lần". `{1, 1, 2}` chính là `{1, 2}`.
- **Multiset** (đa tập): giống set nhưng **nhớ số lần xuất hiện**. Multiset `{1, 1, 2}` khác `{1, 2}`. Thứ tự **không quan trọng**: `{1, 1, 2}` và `{2, 1, 1}` là một.

Trong Python có 3 cách "chứa" một multiset, mỗi cách mạnh ở một việc:

| Cấu trúc | Ví dụ với `[1, 1, 2]` | Giữ số trùng? | Mạnh ở việc gì |
|---|---|---|---|
| `list` | `[1, 1, 2]` | Có (lưu từng bản) | truy cập theo **chỉ số** `a[i]` cực nhanh → chọn ngẫu nhiên dễ |
| `set` | `{1, 2}` | **Không** (mất số trùng) | kiểm tra `x in s`, thêm, xóa theo **giá trị** nhanh |
| `Counter` / `dict` đếm | `{1: 2, 2: 1}` | Có (lưu **số lần**) | biết ngay mỗi giá trị xuất hiện mấy lần |

Bài này muốn **cả** "chọn theo chỉ số" lẫn "tìm theo giá trị" — gợi ý rằng ta sẽ phải **kết hợp** nhiều cấu trúc.

Lưu ý quan trọng: vì multiset **không quan tâm thứ tự**, ta được phép **xáo trộn** vị trí các phần tử trong `list` tùy ý. Đây chính là "giấy phép" cho mẹo đổi chỗ ở mục 3.

In [2]:
a = [1, 1, 2]
s = set(a)
c = Counter(a)
print("list   :", a, "| len =", len(a))
print("set    :", s, "| len =", len(s), "<- mất một số 1")
print("Counter:", dict(c), "| tổng số phần tử =", sum(c.values()))

# Multiset không quan tâm thứ tự: [1,1,2] và [2,1,1] là "cùng một túi"
print(Counter([1, 1, 2]) == Counter([2, 1, 1]))   # True
print([1, 1, 2] == [2, 1, 1])                     # False: list có phân biệt thứ tự

list   : [1, 1, 2] | len = 3
set    : {1, 2} | len = 2 <- mất một số 1
Counter: {1: 2, 2: 1} | tổng số phần tử = 3
True
False


### 2.2 Chi phí thao tác trên `list`: cái nào `O(1)`, cái nào `O(n)`?

Một `list` Python là một **mảng liên tục** trong bộ nhớ: các ô nằm sát nhau, đánh số `0, 1, 2, ...`. Từ đó suy ra chi phí:

| Thao tác | Ví dụ | Chi phí | Vì sao |
|---|---|---|---|
| Đọc / ghi theo chỉ số | `a[i]`, `a[i] = x` | `O(1)` | tính thẳng ra địa chỉ ô thứ `i` |
| Thêm vào cuối | `a.append(x)` | `O(1)` khấu hao* | chỉ ghi vào ô trống ở cuối |
| Bỏ phần tử cuối | `a.pop()` | `O(1)` | chỉ giảm độ dài đi 1 |
| Tìm theo giá trị | `x in a`, `a.index(x)` | `O(n)` | phải dò lần lượt từng ô |
| Xóa ở giữa | `a.pop(i)`, `del a[i]`, `a.remove(x)` | `O(n)` | mọi phần tử **phía sau** phải **dời lên** 1 ô |

\*"Khấu hao" (amortized): thỉnh thoảng list hết chỗ phải xin vùng nhớ lớn hơn và chép sang, nhưng tính trung bình trên nhiều lần `append` thì vẫn `O(1)`.

Hai dòng in đậm trong bảng là gốc rễ của bài: **tìm theo giá trị** và **xóa ở giữa** đều `O(n)`. Ta phải né cả hai.

Ví dụ xóa ở giữa: `a = [10, 20, 30, 40, 50]`, `del a[1]` → `30, 40, 50` đều phải dời sang trái 1 ô → `[10, 30, 40, 50]`. Nếu list có 100 000 phần tử và ta xóa ở đầu, 99 999 phần tử phải dời.

Cell dưới đo thời gian để **thấy** sự khác biệt (con số cụ thể tùy máy, hãy nhìn **tỉ lệ**).

In [3]:
import time

a = [10, 20, 30, 40, 50]
del a[1]
print("sau del a[1]:", a)          # 30, 40, 50 đã dời sang trái

def time_it(f):
    t0 = time.perf_counter()
    f()
    return time.perf_counter() - t0

N = 50_000
big = list(range(N))
t_end = time_it(lambda: [big.pop() for _ in range(N)])      # bỏ phần tử CUỐI: O(1) mỗi lần
big = list(range(N))
t_front = time_it(lambda: [big.pop(0) for _ in range(N)])   # bỏ phần tử ĐẦU: O(n) mỗi lần
print(f"pop() cuối x{N}: {t_end:.4f}s | pop(0) đầu x{N}: {t_front:.4f}s | chậm hơn ≈ {t_front / t_end:.0f} lần")

sau del a[1]: [10, 30, 40, 50]


pop() cuối x50000: 0.0028s | pop(0) đầu x50000: 0.1897s | chậm hơn ≈ 69 lần


### 2.3 Module `random`: `choice`, `randrange`, `seed`

- `random.choice(seq)` chọn **đều** một phần tử của một dãy có chỉ số (list, tuple, string). Bên trong nó chỉ làm: chọn một chỉ số ngẫu nhiên `i` trong `[0, len(seq))` rồi trả `seq[i]` → `O(1)`.
- `random.randrange(n)` trả một số nguyên ngẫu nhiên đều trong `0, 1, ..., n-1`.
- `random.seed(s)` đặt "hạt giống" cho bộ sinh số ngẫu nhiên. Cùng hạt giống → **cùng dãy số ngẫu nhiên** ở mọi lần chạy. Ta dùng nó để notebook chạy lại vẫn ra đúng output như lúc viết, và để test có thể lặp lại.

**Bẫy:** `random.choice` **không** dùng được trên `set` hay `dict`, vì chúng **không có chỉ số** (không có "phần tử thứ `i`"). Muốn chọn ngẫu nhiên từ set phải đổi sang list trước — `random.choice(list(s))` — và việc đổi này tốn `O(n)`. Đây là lý do `getRandom` bắt buộc cần một `list`.

In [4]:
import random

random.seed(381)
print([random.choice([1, 1, 2]) for _ in range(10)])
print([random.randrange(3) for _ in range(10)])

random.seed(381)                                    # đặt lại cùng seed ...
print([random.choice([1, 1, 2]) for _ in range(10)])   # ... ra lại y hệt dãy đầu tiên

try:
    random.choice({1, 2, 3})                       # set không có chỉ số
except TypeError as e:
    print("TypeError:", e)

[1, 1, 2, 1, 1, 1, 1, 2, 1, 2]
[2, 2, 2, 2, 1, 2, 0, 1, 2, 0]
[1, 1, 2, 1, 1, 1, 1, 2, 1, 2]
TypeError: 'set' object is not subscriptable


### 2.4 Mấu chốt xác suất: chọn đều **một vị trí** → tự động tỉ lệ với số lần xuất hiện

Đề đòi xác suất của giá trị `v` bằng `c(v) / n` (số lần `v` xuất hiện chia tổng số phần tử). Nghe có vẻ phải tính toán phức tạp, nhưng thật ra có một cách cực đơn giản:

> Nếu ta lưu **từng bản sao** của mỗi giá trị vào một list (số `1` xuất hiện 2 lần thì list có 2 ô chứa `1`), rồi **chọn đều một ô**, thì xác suất ra giá trị `v` = (số ô chứa `v`) / (tổng số ô) = `c(v) / n`.

Ví dụ `[1, 1, 2]`: có 3 ô, chọn đều mỗi ô với xác suất `1/3`. Hai ô chứa `1` → `P(1) = 1/3 + 1/3 = 2/3`. Một ô chứa `2` → `P(2) = 1/3`. Đúng yêu cầu đề!

Vậy `getRandom` chỉ cần `random.choice(self.nums)` với `self.nums` chứa **đủ mọi bản sao**. Toàn bộ độ khó dồn vào việc giữ cho `insert` và `remove` cũng `O(1)` trên list đó.

Kiểm chứng bằng **thực nghiệm**: gọi 60 000 lần rồi dùng `Counter` đếm tần suất.

In [5]:
random.seed(1)
TRIALS = 60_000
freq = Counter(random.choice([1, 1, 2]) for _ in range(TRIALS))
for v in sorted(freq):
    print(f"giá trị {v}: {freq[v]:>6} lần -> tần suất {freq[v] / TRIALS:.4f} | lý thuyết {Counter([1, 1, 2])[v] / 3:.4f}")

giá trị 1:  40122 lần -> tần suất 0.6687 | lý thuyết 0.6667
giá trị 2:  19878 lần -> tần suất 0.3313 | lý thuyết 0.3333


### 2.5 Class, trạng thái, và hàm chạy chuỗi lệnh kiểu LeetCode

Nhắc nhanh về class:

- `__init__(self)` chạy **đúng 1 lần** khi tạo object; đây là chỗ tạo các "ngăn chứa" như `self.nums = []`.
- Các method khác đọc / sửa `self.xxx`, nên dữ liệu **còn nguyên** giữa các lần gọi `insert`, `remove`, `getRandom`.

Để test giống LeetCode, ta viết 3 hàm tiện ích, dùng chung cho **mọi** cách cài đặt trong notebook:

1. `run_commands(cls, ops, args)`: tạo `obj = cls(*args[0])`, rồi với mỗi lệnh gọi `getattr(obj, op)(*a)` và gom kết quả. `getattr(obj, "insert")` lấy method theo **tên dạng chuỗi**; `*a` "rải" list tham số thành đối số (`f(*[1])` = `f(1)`, `f(*[])` = `f()`).
2. `show(outputs)`: in giống LeetCode (`None` → `null`, `True` → `true`, `False` → `false`).
3. `check_run(cls, ops, args, seed)`: đặt `random.seed(seed)`, chạy chuỗi lệnh, đồng thời giữ một **"trọng tài"** (oracle) là `Counter` mô phỏng multiset đúng. Với `insert` / `remove` nó so **chính xác** giá trị `True/False`; với `getRandom` nó kiểm tra kết quả **thuộc** multiset ở thời điểm đó (`oracle[out] > 0`). Sai ở đâu là `assert` báo ngay.

Vì `getRandom` ngẫu nhiên nên ta không so với `[null, true, false, true, 2, true, 1]` từng số một — chỉ các vị trí `insert` / `remove` là cố định.

In [6]:
def run_commands(cls, ops, args):
    # Chạy chuỗi lệnh kiểu LeetCode: ops[0] là lệnh tạo object
    obj = cls(*args[0])
    outputs = [None]                          # lời gọi khởi tạo -> null
    for op, a in zip(ops[1:], args[1:]):
        outputs.append(getattr(obj, op)(*a))  # ví dụ getattr(obj, "insert")(1)
    return outputs


def show(outputs):
    # In giống LeetCode: None -> null, True -> true, False -> false
    def fmt(x):
        if x is None:
            return "null"
        if isinstance(x, bool):
            return "true" if x else "false"
        return str(x)
    return "[" + ", ".join(fmt(x) for x in outputs) + "]"


def check_run(cls, ops, args, seed=0):
    # Chạy + kiểm tra bằng một Counter "trọng tài" mô phỏng multiset đúng
    random.seed(seed)
    outputs = run_commands(cls, ops, args)
    oracle = Counter()
    for op, a, out in zip(ops[1:], args[1:], outputs[1:]):
        if op == "insert":
            expected = oracle[a[0]] == 0      # True nếu TRƯỚC khi thêm chưa có
            oracle[a[0]] += 1
            assert out == expected, (op, a, out, expected)
        elif op == "remove":
            expected = oracle[a[0]] > 0       # True nếu đang có mặt
            if expected:
                oracle[a[0]] -= 1
            assert out == expected, (op, a, out, expected)
        elif op == "getRandom":
            assert oracle[out] > 0, ("getRandom trả về giá trị không có trong collection", out, +oracle)
    return outputs


ops1 = ["RandomizedCollection", "insert", "insert", "insert", "getRandom", "remove", "getRandom"]
args1 = [[], [1], [1], [2], [], [1], []]
print(show([None, True, False, True, 2, True, 1]))   # output mẫu của đề

[null, true, false, true, 2, true, 1]


## 3. Ôn bài 380 — Insert Delete GetRandom O(1) (không có số trùng)

### 3.1 Đề bài 380 và ý tưởng

Bài 380 (`RandomizedSet`) giống hệt bài này nhưng **không cho trùng**: `insert(val)` trả `false` và **không thêm** nếu `val` đã có; `remove(val)` xóa `val` nếu có. Ba thao tác đều phải `O(1)` trung bình.

Lời giải kinh điển dùng **hai** cấu trúc chạy song song:

- `self.nums` — một `list` chứa các giá trị (để `getRandom` chọn chỉ số ngẫu nhiên `O(1)`);
- `self.idx` — một `dict` **giá trị → chỉ số** của giá trị đó trong `self.nums` (để biết ngay `val` nằm ở ô nào, `O(1)`).

Ví dụ: `nums = [5, 8, 3]` thì `idx = {5: 0, 8: 1, 3: 2}`.

Vấn đề còn lại: **xóa** `nums[i]` ở giữa list tốn `O(n)` (mục 2.2). Mẹo giải quyết ở mục kế tiếp.

### 3.2 Mẹo "đổi chỗ với phần tử cuối rồi `pop`"

Vì multiset (và set) **không quan tâm thứ tự**, muốn xóa ô `i` ta làm:

1. Chép phần tử **cuối** đè lên ô `i`: `a[i] = a[-1]`.
2. Bỏ ô cuối: `a.pop()` — `O(1)`.

Không phần tử nào phải dời chỗ, chỉ **một** phần tử (phần tử cuối) bị "chuyển nhà" sang ô `i`.

Ví dụ `a = [5, 8, 3, 9]`, xóa ô `i = 1` (số `8`):

| Bước | Thao tác | `a` |
|---|---|---|
| 0 | ban đầu | `[5, 8, 3, 9]` |
| 1 | `a[1] = a[-1]` (chép `9` đè lên `8`) | `[5, 9, 3, 9]` |
| 2 | `a.pop()` | `[5, 9, 3]` |

Thứ tự thay đổi (`9` nhảy lên trước `3`), nhưng **túi** vẫn đúng: `{5, 9, 3}`.

Trường hợp đặc biệt: nếu `i` **chính là ô cuối** thì bước 1 là "chép chính nó đè lên chính nó" (không đổi gì), bước 2 bỏ nó đi — vẫn đúng. Ghi nhớ trường hợp này, nó sẽ quay lại làm khó ta ở bài 381.

In [7]:
def remove_at(a, i):
    # Xóa ô i trong O(1) bằng cách đổi chỗ với phần tử cuối (KHÔNG giữ thứ tự)
    a[i] = a[-1]
    a.pop()


a = [5, 8, 3, 9]
remove_at(a, 1)
print("xóa ô 1   :", a)      # [5, 9, 3]

b = [5, 8, 3, 9]
remove_at(b, 3)              # ô cần xóa CHÍNH LÀ ô cuối
print("xóa ô cuối:", b)      # [5, 8, 3]

c = [7]
remove_at(c, 0)              # list chỉ có 1 phần tử
print("xóa ô duy nhất:", c)  # []

xóa ô 1   : [5, 9, 3]
xóa ô cuối: [5, 8, 3]
xóa ô duy nhất: []


### 3.3 Code `RandomizedSet` (bài 380)

Khi phần tử cuối bị "chuyển nhà" sang ô `i`, ta phải **cập nhật địa chỉ** của nó trong `dict`: `idx[last] = i`. Thứ tự các dòng trong `remove`:

```text
i = idx[val]            # val đang ở ô i
last = nums[-1]         # phần tử cuối
nums[i] = last          # chuyển last vào ô i
idx[last] = i           # cập nhật địa chỉ mới của last
nums.pop()              # bỏ ô cuối
del idx[val]            # val không còn nữa
```

Để ý: `del idx[val]` đặt **sau** `idx[last] = i`. Nếu `val` chính là phần tử cuối (`val == last`), dòng `idx[last] = i` sẽ ghi `idx[val]`, rồi dòng `del` xóa nó đi — kết quả đúng. Nếu đảo hai dòng, ta sẽ xóa xong lại "hồi sinh" `idx[val]`. Nguyên tắc **"cập nhật trước, xóa sau"** này sẽ quay lại y hệt ở bài 381.

In [8]:
class RandomizedSet:
    def __init__(self):
        self.nums = []          # các giá trị (để chọn ngẫu nhiên)
        self.idx = {}           # giá trị -> chỉ số trong nums

    def insert(self, val: int) -> bool:
        if val in self.idx:
            return False
        self.idx[val] = len(self.nums)
        self.nums.append(val)
        return True

    def remove(self, val: int) -> bool:
        if val not in self.idx:
            return False
        i = self.idx[val]
        last = self.nums[-1]
        self.nums[i] = last
        self.idx[last] = i      # cập nhật TRƯỚC
        self.nums.pop()
        del self.idx[val]       # xóa SAU
        return True

    def getRandom(self) -> int:
        return random.choice(self.nums)


# Example 1 của bài 380
ops380 = ["RandomizedSet", "insert", "remove", "insert", "getRandom", "remove", "insert", "getRandom"]
args380 = [[], [1], [2], [2], [], [1], [2], []]
random.seed(0)
print(show(run_commands(RandomizedSet, ops380, args380)))   # đề: [null, true, false, true, 2, true, false, 2]

rs = RandomizedSet()
for v in [5, 8, 3, 9]:
    rs.insert(v)
rs.remove(8)
print("nums =", rs.nums, "| idx =", rs.idx)                # 9 chuyển vào ô 1, idx[9] = 1

[null, true, false, true, 2, true, false, 2]
nums = [5, 9, 3] | idx = {5: 0, 3: 2, 9: 1}


### 3.4 Vì sao không bê nguyên bài 380 sang bài 381?

Ở bài 380, mỗi giá trị chỉ nằm ở **một** ô nên `dict` lưu **một** chỉ số là đủ. Ở bài 381, số `1` có thể nằm ở ô `0` **và** ô `1` cùng lúc. Nếu cứ để `dict` lưu một chỉ số, lần `insert(1)` thứ hai sẽ **ghi đè** chỉ số cũ → ta "quên" mất một bản sao.

Cell dưới thử "vá" bài 380 bằng cách luôn thêm (bỏ điều kiện `if val in self.idx`). Sau `insert(1), insert(1), insert(2)` ta có `nums = [1, 1, 2]` nhưng `idx = {1: 1, 2: 2}` — ô `0` không còn ai trỏ tới. Rồi `remove(1)` hai lần: lần hai trả `False` dù vẫn còn một số `1` nằm trong `nums`!

**Bài học:** với số trùng, mỗi giá trị cần nhớ **cả một tập các chỉ số**. Câu hỏi "tập đó nên là `list` hay `set`?" sẽ được trả lời ở mục 6.

In [9]:
class NaiveDuplicateSet(RandomizedSet):
    def insert(self, val: int) -> bool:
        was_new = val not in self.idx
        self.idx[val] = len(self.nums)     # GHI ĐÈ chỉ số cũ nếu val đã có!
        self.nums.append(val)
        return was_new


bad = NaiveDuplicateSet()
for v in [1, 1, 2]:
    bad.insert(v)
print("nums =", bad.nums, "| idx =", bad.idx, "<- ô 0 bị 'mồ côi'")
print("remove(1) lần 1:", bad.remove(1), "| nums =", bad.nums, "| idx =", bad.idx)
print("remove(1) lần 2:", bad.remove(1), "| nums =", bad.nums, "<- vẫn còn số 1 mà trả False!")

nums = [1, 1, 2] | idx = {1: 1, 2: 2} <- ô 0 bị 'mồ côi'
remove(1) lần 1: True | nums = [1, 2] | idx = {2: 1}
remove(1) lần 2: False | nums = [1, 2] <- vẫn còn số 1 mà trả False!


## 4. Cách 1 — Chỉ dùng một `list`

Cách người mới nghĩ ra đầu tiên: *"Cứ bỏ hết vào một list. List giữ được số trùng, `random.choice` chọn đều một ô là đúng phân phối luôn."*

- `insert(val)`: kiểm tra `val in self.items` (để biết trả `True` hay `False`), rồi `append`.
- `remove(val)`: nếu có thì `self.items.remove(val)` — hàm có sẵn, xóa lần xuất hiện **đầu tiên** của `val`.
- `getRandom()`: `random.choice(self.items)`.

Cách này **chắc chắn đúng** (list chính là multiset với đủ mọi bản sao, mục 2.4), nên ta sẽ dùng nó làm **baseline** để đối chiếu các cách nhanh hơn.

In [10]:
class RandomizedCollectionList:
    def __init__(self):
        self.items = []

    def insert(self, val: int) -> bool:
        was_new = val not in self.items      # O(n): dò từng ô
        self.items.append(val)               # O(1)
        return was_new

    def remove(self, val: int) -> bool:
        if val not in self.items:            # O(n)
            return False
        self.items.remove(val)               # O(n): tìm + dời các phần tử phía sau
        return True

    def getRandom(self) -> int:
        return random.choice(self.items)     # O(1)


out = check_run(RandomizedCollectionList, ops1, args1, seed=0)
print("Example 1:", show(out))

Example 1: [null, true, false, true, 1, true, 2]


### Độ phức tạp cách 1 và chỗ vướng

- `insert`: `O(n)` vì `val not in self.items` phải dò cả list.
- `remove`: `O(n)` vì phải **tìm** `val` rồi **dời** các phần tử phía sau lên.
- `getRandom`: `O(1)`.
- Bộ nhớ: `O(n)`.

Trường hợp xấu: `2·10^5` lời gọi, ví dụ `10^5` lần `insert` số khác nhau rồi `10^5` lần `remove`. Mỗi lần tốn cỡ `n ≈ 10^5` bước → tổng cỡ `10^5 · 10^5 = 10^10` bước. Python làm khoảng `10^7`–`10^8` bước đơn giản mỗi giây → mất hàng phút → **TLE** (Time Limit Exceeded). Mục 13 sẽ đo thời gian thật.

**Nút thắt:** list chỉ giỏi trả lời "ô thứ `i` chứa gì?", còn câu hỏi "giá trị `val` nằm ở ô nào?" thì phải dò từ đầu. Ta cần một cấu trúc trả lời câu hỏi **theo giá trị** trong `O(1)` → **hash** (`dict` / `set`).

## 5. Cách 2 — `dict` đếm số lần xuất hiện

### 5.1 Nền tảng: `dict` đếm và `Counter`

Một `dict` (từ điển) lưu các cặp **key → value**, tra cứu / thêm / xóa theo key **trung bình `O(1)`** nhờ **hash** (băm): Python biến key thành một con số (hash) rồi dùng nó để nhảy thẳng tới "ngăn" chứa key, không cần dò tuần tự.

Dùng `dict` để **đếm**: key là giá trị, value là số lần xuất hiện. Với `[1, 1, 2]` → `{1: 2, 2: 1}`.

- `d.get(x, 0)` đọc số đếm của `x`, nếu chưa có thì trả `0` (không lỗi).
- `del d[x]` xóa hẳn key `x`.
- `collections.Counter` là một `dict` đếm có sẵn: `Counter([1, 1, 2])` → `Counter({1: 2, 2: 1})`, và `c[x]` trả `0` nếu `x` chưa có.

In [11]:
d = {}
for v in [1, 1, 2]:
    d[v] = d.get(v, 0) + 1
print("dict đếm:", d)                       # {1: 2, 2: 1}
print("1 có mặt?", 1 in d, "| số lần:", d[1])
d[1] -= 1                                    # xóa một số 1
print("sau khi bớt một số 1:", d)
d[2] -= 1
if d[2] == 0:
    del d[2]                                 # hết thì xóa hẳn key cho gọn
print("sau khi bớt số 2:", d, "| 2 có mặt?", 2 in d)

c = Counter([1, 1, 2])
print(c, "| c[99] =", c[99])                 # Counter trả 0 cho key chưa có

dict đếm: {1: 2, 2: 1}
1 có mặt? True | số lần: 2
sau khi bớt một số 1: {1: 1, 2: 1}
sau khi bớt số 2: {1: 1} | 2 có mặt? False
Counter({1: 2, 2: 1}) | c[99] = 0


### 5.2 Cách 2: `insert` / `remove` nhanh, nhưng `getRandom` thì sao?

Với `dict` đếm `self.count` và tổng số phần tử `self.total`:

- `insert(val)`: tăng `count[val]`; trả `True` nếu sau khi tăng `count[val] == 1` (tức là trước đó chưa có). `O(1)`.
- `remove(val)`: nếu `val` không có → `False`; ngược lại giảm `count[val]`, về `0` thì xóa key. `O(1)`.
- `getRandom()`: **không có list** để chọn chỉ số! Ta phải "tưởng tượng" ra list `[1, 1, 2]` mà không thật sự tạo nó:
  1. Chọn `r = random.randrange(total)` — một **vị trí ảo** trong list tưởng tượng.
  2. Đi qua từng giá trị: giá trị `v` chiếm `count[v]` vị trí liên tiếp. Nếu `r < count[v]` thì vị trí `r` rơi vào "đoạn" của `v` → trả `v`. Ngược lại trừ `r -= count[v]` và sang giá trị kế.

Ví dụ `count = {1: 2, 2: 1}`, list tưởng tượng `[1, 1, 2]`:

| `r` | Xét `v = 1` (`count = 2`) | Xét `v = 2` (`count = 1`) | Kết quả |
|---:|---|---|---:|
| 0 | `0 < 2` → trả 1 | | `1` |
| 1 | `1 < 2` → trả 1 | | `1` |
| 2 | `2 >= 2` → `r = 0` | `0 < 1` → trả 2 | `2` |

Mỗi vị trí ảo `r` ứng với đúng một ô của list tưởng tượng, nên phân phối vẫn đúng `c(v) / n`.

In [12]:
class RandomizedCollectionCounter:
    def __init__(self):
        self.count = {}       # giá trị -> số lần xuất hiện
        self.total = 0        # tổng số phần tử (đếm cả trùng)

    def insert(self, val: int) -> bool:
        self.count[val] = self.count.get(val, 0) + 1
        self.total += 1
        return self.count[val] == 1

    def remove(self, val: int) -> bool:
        if val not in self.count:
            return False
        self.count[val] -= 1
        if self.count[val] == 0:
            del self.count[val]
        self.total -= 1
        return True

    def pick_by_rank(self, r):
        # Trả về giá trị ở "vị trí ảo" r của list tưởng tượng: O(k)
        for val, c in self.count.items():
            if r < c:
                return val
            r -= c

    def getRandom(self) -> int:
        return self.pick_by_rank(random.randrange(self.total))


rc = RandomizedCollectionCounter()
for v in [1, 1, 2]:
    rc.insert(v)
print("count =", rc.count, "| total =", rc.total)
for r in range(rc.total):
    print(f"  r = {r} -> {rc.pick_by_rank(r)}")

print("Example 1:", show(check_run(RandomizedCollectionCounter, ops1, args1, seed=0)))

count = {1: 2, 2: 1} | total = 3
  r = 0 -> 1
  r = 1 -> 1
  r = 2 -> 2
Example 1: [null, true, false, true, 1, true, 2]


### Độ phức tạp cách 2 và chỗ vướng

- `insert`, `remove`: `O(1)` trung bình. Tốt hơn hẳn cách 1.
- `getRandom`: `O(k)` — phải đi qua các giá trị khác nhau cho tới khi gặp "đoạn" chứa `r`. Khi mọi số đều khác nhau thì `k = n`, tức là `O(n)`.
- Bộ nhớ: `O(k)`.

Ta chỉ đổi chỗ nút thắt: cách 1 chậm ở `insert` / `remove`, cách 2 chậm ở `getRandom`. Nếu đề gọi `10^5` lần `insert` số khác nhau rồi `10^5` lần `getRandom`, ta lại có `~10^10` bước.

**Bài học từ hai cách:**

| Nhu cầu | Cấu trúc giỏi việc đó |
|---|---|
| chọn ngẫu nhiên đều theo **vị trí** trong `O(1)` | `list` chứa đủ mọi bản sao (cách 1) |
| tìm / thêm / xóa **theo giá trị** trong `O(1)` | hash: `dict` / `set` (cách 2) |

→ Ý tưởng: **dùng cả hai cùng lúc**, giống bài 380, và giữ chúng **luôn khớp nhau**.

## 6. Bước đột phá — `list` + `dict` giá trị → **tập các chỉ số**

Giữ lại `self.nums` (list chứa đủ mọi bản sao, cho `getRandom`) và thêm `self.pos`: với mỗi giá trị `v`, `pos[v]` là **tập tất cả các chỉ số** `i` mà `nums[i] == v`.

Ví dụ `nums = [1, 1, 2, 1]` → `pos = {1: {0, 1, 3}, 2: {2}}`.

Hãy liệt kê **chính xác** những việc mà `remove(val)` (dùng mẹo đổi chỗ ở mục 3.2) cần làm với "tập chỉ số":

1. **Lấy ra một chỉ số bất kỳ** `i` của `val` (xóa bản nào cũng được, vì mọi bản `val` như nhau) → lấy và bỏ khỏi `pos[val]`.
2. Phần tử cuối `last = nums[n-1]` chuyển vào ô `i` → **thêm** `i` vào `pos[last]`.
3. Ô `n-1` biến mất → **xóa đúng chỉ số `n-1`** khỏi `pos[last]`. Đây là xóa một phần tử **cụ thể** (không phải "bất kỳ") khỏi tập.

Còn `insert` thì chỉ cần **thêm** chỉ số `n` vào `pos[val]`.

Vậy "tập chỉ số" phải làm nhanh 3 việc: **thêm**, **lấy ra một phần tử bất kỳ**, và **xóa một phần tử cụ thể**. Nếu dùng `list` thì việc 3 phải dò tìm `n-1` trong list → `O(c)`. Dùng **`set`** thì cả ba đều `O(1)` trung bình. Ta học `set` trước rồi làm thí nghiệm so sánh ở mục 6.3.

### 6.1 Nền tảng: `set` trong Python

`set` là "túi" **không trùng**, **không thứ tự**, cài bằng hash giống `dict` (chỉ có key, không có value).

| Thao tác | Ví dụ | Chi phí trung bình | Ghi chú |
|---|---|---|---|
| tạo rỗng | `s = set()` | `O(1)` | **không** viết `{}` — đó là `dict` rỗng! |
| thêm | `s.add(x)` | `O(1)` | có rồi thì không làm gì |
| kiểm tra | `x in s` | `O(1)` | |
| xóa phần tử cụ thể | `s.discard(x)` | `O(1)` | `x` không có thì **im lặng** bỏ qua |
| xóa phần tử cụ thể | `s.remove(x)` | `O(1)` | `x` không có thì **lỗi** `KeyError` |
| lấy ra một phần tử bất kỳ | `s.pop()` | `O(1)` | set rỗng thì lỗi `KeyError`; **không hứa** trả phần tử nào |
| số phần tử | `len(s)` | `O(1)` | |

Hai điểm cần nhớ:

- `discard` an toàn hơn `remove` khi ta không chắc phần tử có mặt. Ở bài này sẽ có lúc ta "xóa `n-1` khỏi `pos[last]`" ngay sau khi vừa lấy nó ra → nó **có thể đã không còn**; `discard` không làm chương trình sập.
- `s.pop()` trả về một phần tử "bất kỳ" theo cách sắp xếp nội bộ của Python. Code đúng **không được dựa** vào việc nó trả phần tử nào.

In [13]:
s = set()
print(type(set()), type({}))         # set rỗng vs dict rỗng
for x in [0, 1, 4, 1]:
    s.add(x)                         # thêm 1 lần nữa: không có gì thay đổi
print("s =", s, "| len =", len(s), "| 4 in s:", 4 in s)

s.discard(4)
s.discard(99)                        # không có 99: im lặng
print("sau discard(4), discard(99):", s)

try:
    s.remove(99)
except KeyError as e:
    print("remove(99) -> KeyError:", e)

x = s.pop()                          # lấy ra MỘT phần tử bất kỳ
print("pop() lấy ra", x, "| còn lại", s)

<class 'set'> <class 'dict'>
s = {0, 1, 4} | len = 3 | 4 in s: True
sau discard(4), discard(99): {0, 1}
remove(99) -> KeyError: 99
pop() lấy ra 0 | còn lại {1}


### 6.2 Nền tảng: `defaultdict(set)` — dict tự tạo set rỗng

Với `pos` thường, mỗi lần `insert` một giá trị **mới** ta phải viết:

```python
if val not in pos:
    pos[val] = set()
pos[val].add(i)
```

`collections.defaultdict(set)` làm gọn việc này: khi truy cập `pos[val]` mà key chưa có, nó **tự tạo** `set()` rỗng rồi trả về. Ta chỉ cần viết `pos[val].add(i)`.

Cẩn thận một tác dụng phụ: **chỉ cần đọc** `pos[val]` (ví dụ trong `if not pos[val]`) cũng làm key `val` được tạo ra với set rỗng. Còn `val in pos` thì **không** tạo key. Vì vậy trong lời giải ta sẽ:

- kiểm tra có mặt bằng `val in self.pos`;
- và **xóa hẳn key** khi set của nó trở nên rỗng, để "`val in self.pos`" luôn đồng nghĩa với "`val` đang có mặt".

In [14]:
from collections import defaultdict

pos = defaultdict(set)
for i, v in enumerate([1, 1, 2, 1]):     # enumerate cho cả chỉ số i và giá trị v
    pos[v].add(i)
print(dict(pos))                         # {1: {0, 1, 3}, 2: {2}}

print("7 in pos:", 7 in pos)             # 'in' KHÔNG tạo key
print("pos[7] =", pos[7])                # ĐỌC pos[7] -> tự tạo set rỗng!
print("7 in pos:", 7 in pos, "<- key 7 đã bị tạo ra")
del pos[7]                               # dọn đi
print(dict(pos))

{1: {0, 1, 3}, 2: {2}}
7 in pos: False
pos[7] = set()
7 in pos: True <- key 7 đã bị tạo ra
{1: {0, 1, 3}, 2: {2}}


### 6.3 Thí nghiệm: vì sao tập chỉ số phải là `set`, không phải `list`?

Giả sử ta dùng `pos[v]` là **list** các chỉ số. Việc 1 (lấy ra bất kỳ) làm được `O(1)` bằng `pos[v].pop()`, việc 2 (thêm) bằng `append` cũng `O(1)`. Nhưng **việc 3** — xóa đúng chỉ số `n-1` khỏi `pos[last]` — buộc phải `pos[last].remove(n-1)`: dò từ đầu list tới khi gặp `n-1` rồi dời các phần tử phía sau → `O(c)` với `c` = số lần `last` xuất hiện.

Trường hợp xấu nhất: **mọi phần tử bằng nhau**, ví dụ `n` số `7`. Khi đó `pos[7] = [0, 1, ..., n-1]` dài tới `n`. Mỗi `remove(7)`: `pop()` lấy ra `n-1` (rẻ), `append(n-1)` thêm lại vào cuối (rẻ), rồi `index(n-1)` phải dò **từ đầu** list mới gặp nó ở cuối → dò gần hết list. Phiên bản dưới **đếm số ô** mà `list.index` phải dò. Phiên bản `set` thì mỗi việc chỉ đụng 1 phần tử.

Quan sát: khi `n` gấp đôi, tổng số ô phải dò của bản `list` tăng khoảng **4 lần** → tổng `O(n^2)`, mỗi `remove` `O(n)`.

In [15]:
class IndexListDemo:
    # pos[v] là LIST chỉ số; đếm số ô phải dò khi xóa một chỉ số cụ thể
    def __init__(self):
        self.nums, self.pos, self.scanned = [], defaultdict(list), 0

    def insert(self, val):
        self.pos[val].append(len(self.nums))
        self.nums.append(val)

    def remove(self, val):
        lst = self.pos[val]
        i = lst.pop()                         # lấy chỉ số ở cuối list: O(1)
        last_i = len(self.nums) - 1
        last = self.nums[last_i]
        self.nums[i] = last
        self.pos[last].append(i)
        k = self.pos[last].index(last_i)      # DÒ tìm last_i: tốn k + 1 ô
        self.scanned += k + 1
        del self.pos[last][k]
        self.nums.pop()


for n in [1000, 2000, 4000]:
    demo = IndexListDemo()
    for _ in range(n):
        demo.insert(7)                        # mọi phần tử đều là 7
    for _ in range(n):
        demo.remove(7)
    print(f"n = {n}: list chỉ số phải dò tổng cộng {demo.scanned:>9,} ô (≈ {demo.scanned / n:,.0f} ô mỗi remove)")
print("Với set: mỗi remove chỉ pop / add / discard đúng 1 phần tử -> O(1) trung bình.")

n = 1000: list chỉ số phải dò tổng cộng   500,500 ô (≈ 500 ô mỗi remove)
n = 2000: list chỉ số phải dò tổng cộng 2,001,000 ô (≈ 1,000 ô mỗi remove)
n = 4000: list chỉ số phải dò tổng cộng 8,002,000 ô (≈ 2,000 ô mỗi remove)
Với set: mỗi remove chỉ pop / add / discard đúng 1 phần tử -> O(1) trung bình.


## 7. Cách 3 — `list` + `dict` giá trị → `set` chỉ số (cách chính)

### 7.1 Trạng thái và `insert`

Trạng thái:

- `self.nums`: list chứa **mọi bản sao** (thứ tự tùy ý).
- `self.pos`: `defaultdict(set)`, `pos[v]` = tập chỉ số `i` sao cho `nums[i] == v`. Key nào có set rỗng thì **xóa hẳn**.

`insert(val)`:

```python
self.pos[val].add(len(self.nums))   # bản sao mới sẽ nằm ở ô n (ô ngay sau ô cuối hiện tại)
self.nums.append(val)
return len(self.pos[val]) == 1      # sau khi thêm mà chỉ có 1 chỉ số -> trước đó chưa có
```

Ví dụ: `nums = [1, 1]`, `pos = {1: {0, 1}}`. Gọi `insert(2)`: `n = 2` → `pos[2] = {2}`, `nums = [1, 1, 2]`, `len(pos[2]) == 1` → trả `True`. Gọi tiếp `insert(1)`: `pos[1] = {0, 1, 3}` → `len == 3` → trả `False`.

`getRandom()` đơn giản là `random.choice(self.nums)` (mục 2.4).

### 7.2 `remove(val)` từng dòng

```python
def remove(self, val):
    if val not in self.pos:            # (0) không có -> False
        return False
    i = self.pos[val].pop()            # (1) lấy ra MỘT chỉ số bất kỳ của val: xóa bản ở ô i
    last_i = len(self.nums) - 1        #     chỉ số ô cuối
    last = self.nums[last_i]           #     giá trị ở ô cuối
    self.nums[i] = last                # (2) chuyển last vào ô i
    self.pos[last].add(i)              # (3) last giờ có thêm địa chỉ i      <- ADD TRƯỚC
    self.pos[last].discard(last_i)     # (4) ô cuối sắp biến mất            <- DISCARD SAU
    self.nums.pop()                    # (5) bỏ ô cuối
    if not self.pos[val]:              # (6) val hết bản sao -> xóa key
        del self.pos[val]
    return True
```

Ba tình huống của `remove` — hãy tự kiểm tra từng dòng:

| Tình huống | Điều kiện | Điều xảy ra ở (3), (4) |
|---|---|---|
| **A. Bình thường** | `i != last_i`, `last != val` | `pos[last]`: thêm `i`, bỏ `last_i` → địa chỉ của `last` được "chuyển nhà" đúng |
| **B. Xóa chính phần tử cuối** | `i == last_i` (khi đó chắc chắn `last == val`) | (1) đã bỏ `last_i` khỏi `pos[val]`; (3) **thêm lại** `last_i`; (4) **bỏ** `last_i` lần nữa → cuối cùng `last_i` không còn. Đúng! |
| **C. Giá trị trùng với phần tử cuối** | `i != last_i` nhưng `last == val` | `pos[last]` chính là `pos[val]`: (1) bỏ `i`, (3) thêm lại `i`, (4) bỏ `last_i` → về bản chất ta đã xóa bản ở **ô cuối** |

**Vì sao `add` phải đứng trước `discard`?** Hãy xem tình huống B nếu đảo thứ tự: (1) bỏ `last_i`; (4') `discard(last_i)` — không có gì để bỏ; (3') `add(i)` với `i == last_i` → `last_i` **quay lại** `pos[val]`. Sau `nums.pop()`, ô `last_i` không còn tồn tại nhưng `pos[val]` vẫn trỏ tới nó → chỉ số "ma". Mục 14 sẽ chạy thử lỗi này.

Một cách nhớ: **"thêm địa chỉ mới trước, gạch địa chỉ cũ sau"** — giống bài 380: "cập nhật trước, xóa sau".

In [16]:
class RandomizedCollectionSets:
    def __init__(self):
        self.nums = []                     # mọi bản sao (để getRandom)
        self.pos = defaultdict(set)        # giá trị -> tập chỉ số trong nums

    def insert(self, val: int) -> bool:
        self.pos[val].add(len(self.nums))
        self.nums.append(val)
        return len(self.pos[val]) == 1

    def remove(self, val: int) -> bool:
        if val not in self.pos:
            return False
        i = self.pos[val].pop()            # xóa bản sao ở ô i
        last_i = len(self.nums) - 1
        last = self.nums[last_i]
        self.nums[i] = last                # chuyển phần tử cuối vào ô i
        self.pos[last].add(i)              # ADD trước ...
        self.pos[last].discard(last_i)     # ... DISCARD sau
        self.nums.pop()
        if not self.pos[val]:
            del self.pos[val]
        return True

    def getRandom(self) -> int:
        return random.choice(self.nums)


out = check_run(RandomizedCollectionSets, ops1, args1, seed=0)
print("Example 1:", show(out))

demo = RandomizedCollectionSets()
for v in [1, 1, 2]:
    demo.insert(v)
print("nums =", demo.nums, "| pos =", dict(demo.pos))

Example 1: [null, true, false, true, 1, true, 1]
nums = [1, 1, 2] | pos = {1: {0, 1}, 2: {2}}


## 8. Trace từng bước

### 8.1 Kịch bản trace

Ta chọn một chuỗi lệnh "khó" để đi qua **cả 3 tình huống** của `remove`, cộng thêm một lần `remove` giá trị không có và một lần `insert` giá trị đã có:

```text
insert 1, insert 1, insert 2, insert 3, insert 1     -> nums = [1, 1, 2, 3, 1]
remove 1      (val trùng với phần tử cuối: tình huống C hoặc B)
remove 3      (3 đang ở ô cuối: tình huống B)
remove 1      (tình huống A)
remove 5      (không có -> False)
insert 2      (đã có -> False)
```

Hàm `trace_collection` dưới đây làm **đúng** các bước của `RandomizedCollectionSets.remove`, nhưng in ra `i`, `last_i`, `last`, tình huống và trạng thái `nums` / `pos` sau mỗi lệnh. Cuối cùng nó `assert` trạng thái cuối trùng với khi chạy class thật trên cùng chuỗi lệnh.

In [17]:
def fmt_pos(pos):
    # In pos gọn: {1:{0,1}, 2:{2}}
    return "{" + ", ".join(f"{v}:{{{','.join(map(str, sorted(s)))}}}" for v, s in sorted(pos.items())) + "}"


def fmt_set(s):
    return "{" + ",".join(map(str, sorted(s))) + "}"


def trace_collection(ops, args):
    nums, pos = [], defaultdict(set)
    for op, (val,) in zip(ops, args):
        if op == "insert":
            pos[val].add(len(nums))
            nums.append(val)
            res = len(pos[val]) == 1
            print(f"insert({val}) -> {res!s:<5} | nums={nums} pos={fmt_pos(pos)}")
            continue
        if val not in pos:
            print(f"remove({val}) -> False | {val} không có mặt")
            continue
        i = pos[val].pop()
        last_i = len(nums) - 1
        last = nums[last_i]
        case = "B: xóa chính ô cuối" if i == last_i else ("C: val trùng phần tử cuối" if last == val else "A: bình thường")
        print(f"remove({val}): pop i={i} khỏi pos[{val}]; last_i={last_i}, last={last}  [{case}]")
        nums[i] = last
        pos[last].add(i)
        print(f"    nums[{i}]={last}; pos[{last}].add({i})     -> pos[{last}]={fmt_set(pos[last])}")
        pos[last].discard(last_i)
        print(f"    pos[{last}].discard({last_i})  -> pos[{last}]={fmt_set(pos[last])}")
        nums.pop()
        if not pos[val]:
            del pos[val]
        print(f"    -> True  | nums={nums} pos={fmt_pos(pos)}")
    return nums, pos


trace_ops = ["insert"] * 5 + ["remove"] * 4 + ["insert"]
trace_args = [[1], [1], [2], [3], [1], [1], [3], [1], [5], [2]]
t_nums, t_pos = trace_collection(trace_ops, trace_args)

real = RandomizedCollectionSets()
for op, (val,) in zip(trace_ops, trace_args):
    getattr(real, op)(val)
assert real.nums == t_nums and dict(real.pos) == dict(t_pos)
print("Trạng thái cuối khớp với class thật.")

insert(1) -> True  | nums=[1] pos={1:{0}}
insert(1) -> False | nums=[1, 1] pos={1:{0,1}}
insert(2) -> True  | nums=[1, 1, 2] pos={1:{0,1}, 2:{2}}
insert(3) -> True  | nums=[1, 1, 2, 3] pos={1:{0,1}, 2:{2}, 3:{3}}
insert(1) -> False | nums=[1, 1, 2, 3, 1] pos={1:{0,1,4}, 2:{2}, 3:{3}}
remove(1): pop i=0 khỏi pos[1]; last_i=4, last=1  [C: val trùng phần tử cuối]
    nums[0]=1; pos[1].add(0)     -> pos[1]={0,1,4}
    pos[1].discard(4)  -> pos[1]={0,1}
    -> True  | nums=[1, 1, 2, 3] pos={1:{0,1}, 2:{2}, 3:{3}}
remove(3): pop i=3 khỏi pos[3]; last_i=3, last=3  [B: xóa chính ô cuối]
    nums[3]=3; pos[3].add(3)     -> pos[3]={3}
    pos[3].discard(3)  -> pos[3]={}
    -> True  | nums=[1, 1, 2] pos={1:{0,1}, 2:{2}}
remove(1): pop i=1 khỏi pos[1]; last_i=2, last=2  [A: bình thường]
    nums[1]=2; pos[2].add(1)     -> pos[2]={1,2}
    pos[2].discard(2)  -> pos[2]={1}
    -> True  | nums=[1, 2] pos={1:{0}, 2:{1}}
remove(5) -> False | 5 không có mặt
insert(2) -> False | nums=[1, 2, 2] pos={1:{0

### 8.2 Bảng trạng thái (khớp với output ở trên)

| # | Lệnh | `i` lấy ra | `last_i` | `last` | Tình huống | `nums` sau | `pos` sau | Trả về |
|---:|---|---:|---:|---:|---|---|---|---|
| 1 | `insert(1)` | | | | | `[1]` | `{1:{0}}` | `True` |
| 2 | `insert(1)` | | | | | `[1, 1]` | `{1:{0,1}}` | `False` |
| 3 | `insert(2)` | | | | | `[1, 1, 2]` | `{1:{0,1}, 2:{2}}` | `True` |
| 4 | `insert(3)` | | | | | `[1, 1, 2, 3]` | `{1:{0,1}, 2:{2}, 3:{3}}` | `True` |
| 5 | `insert(1)` | | | | | `[1, 1, 2, 3, 1]` | `{1:{0,1,4}, 2:{2}, 3:{3}}` | `False` |
| 6 | `remove(1)` | 0 | 4 | 1 | C | `[1, 1, 2, 3]` | `{1:{0,1}, 2:{2}, 3:{3}}` | `True` |
| 7 | `remove(3)` | 3 | 3 | 3 | B | `[1, 1, 2]` | `{1:{0,1}, 2:{2}}` | `True` |
| 8 | `remove(1)` | 1 | 2 | 2 | A | `[1, 2]` | `{1:{0}, 2:{1}}` | `True` |
| 9 | `remove(5)` | | | | không có | `[1, 2]` | không đổi | `False` |
| 10 | `insert(2)` | | | | | `[1, 2, 2]` | `{1:{0}, 2:{1,2}}` | `False` |

Đọc kỹ từng dòng `remove`:

- **Dòng 6 (C):** `set.pop()` lấy ra `i = 0` từ `{0, 1, 4}`. Phần tử cuối (ô `4`) cũng là `1`. `nums[0] = 1` (không đổi gì về giá trị), `pos[1]`: thêm lại `0` → `{0, 1, 4}`, bỏ `4` → `{0, 1}`. Kết quả giống như ta đã xóa **bản ở ô cuối**. Nếu không có bước `add(0)`, ô `0` vẫn chứa số `1` nhưng không ai nhớ nó → sai.
- **Dòng 7 (B):** `3` nằm đúng ở ô cuối `3`. `pop()` lấy `i = 3`; `add(3)` thêm lại `3`; `discard(3)` bỏ nó → `pos[3]` rỗng → xóa key `3`. Đúng vì ô `3` không còn tồn tại.
- **Dòng 8 (A):** lần này `set.pop()` lấy ra `i = 1` (không phải `0`!). Đó là vì `pop()` của `set` không hứa trả phần tử nào — Python có thể trả phần tử khác tùy lịch sử thao tác trên set. Phần tử cuối là `2` ở ô `2` → chuyển `2` vào ô `1`: `pos[2]` thêm `1`, bỏ `2` → `{1}`.

Dù `pop()` trả chỉ số nào, kết quả **vẫn đúng** (multiset là `{1, 2}`); chỉ có thứ tự trong `nums` là khác nhau. Mục 8.3 chạy thử để chứng tỏ điều đó.

### 8.3 Soi riêng hai tình huống "bẫy" B và C

Ta tự chọn chỉ số bị lấy ra (thay cho `set.pop()`) để **cố ý** tạo ra từng tình huống, rồi kiểm tra trạng thái sau đó **nhất quán**: với mọi ô `i`, `i` phải nằm trong `pos[nums[i]]`, và tổng kích thước các set phải bằng `len(nums)`.

- `[5]`, xóa `5`: chỉ có 1 phần tử, nên chắc chắn là tình huống **B** (`i = last_i = 0`).
- `[1, 2, 1]`, xóa `1` lấy `i = 0`: tình huống **C** (ô cuối cũng là `1`).
- `[1, 2, 1]`, xóa `1` lấy `i = 2`: tình huống **B**.
- `[1, 2, 1]`, xóa `2`: tình huống **A**.

Cả 4 lần đều phải ra multiset đúng.

In [18]:
def consistent(nums, pos):
    # Mọi ô i đều được pos[nums[i]] ghi nhận, không thừa không thiếu, không có set rỗng
    ok_cover = all(i in pos[v] for i, v in enumerate(nums))
    ok_size = sum(len(s) for s in pos.values()) == len(nums)
    ok_nonempty = all(len(s) > 0 for s in pos.values())
    return ok_cover and ok_size and ok_nonempty


def remove_with_choice(nums, pos, val, i):
    # Giống remove của cách 3, nhưng TA chọn chỉ số i thay cho pos[val].pop()
    pos[val].discard(i)
    last_i = len(nums) - 1
    last = nums[last_i]
    nums[i] = last
    pos[last].add(i)
    pos[last].discard(last_i)
    nums.pop()
    if not pos[val]:
        del pos[val]


for start, val, i in [([5], 5, 0), ([1, 2, 1], 1, 0), ([1, 2, 1], 1, 2), ([1, 2, 1], 2, 1)]:
    nums = list(start)
    pos = defaultdict(set)
    for j, v in enumerate(nums):
        pos[v].add(j)
    remove_with_choice(nums, pos, val, i)
    print(f"{start} xóa {val} tại ô {i} -> nums={nums} pos={fmt_pos(pos)} | nhất quán: {consistent(nums, pos)}"
          f" | multiset đúng: {Counter(nums) == Counter(start) - Counter([val])}")

[5] xóa 5 tại ô 0 -> nums=[] pos={} | nhất quán: True | multiset đúng: True
[1, 2, 1] xóa 1 tại ô 0 -> nums=[1, 2] pos={1:{0}, 2:{1}} | nhất quán: True | multiset đúng: True
[1, 2, 1] xóa 1 tại ô 2 -> nums=[1, 2] pos={1:{0}, 2:{1}} | nhất quán: True | multiset đúng: True
[1, 2, 1] xóa 2 tại ô 1 -> nums=[1, 1] pos={1:{0,1}} | nhất quán: True | multiset đúng: True


## 9. Code nộp LeetCode

Chỉ cần đổi tên class thành đúng `RandomizedCollection` và bỏ các phần in debug. LeetCode Python3 đã `import` sẵn nhiều module (`random`, `collections`...), nhưng ta vẫn tự `import` cho chắc chắn và để chạy được ở mọi nơi.

In [19]:
import random
from collections import defaultdict


class RandomizedCollection:

    def __init__(self):
        self.nums = []                     # mọi bản sao
        self.pos = defaultdict(set)        # giá trị -> tập chỉ số trong nums

    def insert(self, val: int) -> bool:
        self.pos[val].add(len(self.nums))
        self.nums.append(val)
        return len(self.pos[val]) == 1

    def remove(self, val: int) -> bool:
        if val not in self.pos:
            return False
        i = self.pos[val].pop()
        last_i = len(self.nums) - 1
        last = self.nums[last_i]
        self.nums[i] = last
        self.pos[last].add(i)              # add trước
        self.pos[last].discard(last_i)     # discard sau
        self.nums.pop()
        if not self.pos[val]:
            del self.pos[val]
        return True

    def getRandom(self) -> int:
        return random.choice(self.nums)

### 9.1 Bộ test cho `RandomizedCollection`

Mỗi ca test là một chuỗi lệnh; `check_run` kiểm tra mọi `True/False` và mọi kết quả `getRandom`. Ngoài ra ta so **chính xác** phần `insert` / `remove` với kết quả mong đợi viết tay (bỏ qua vị trí `getRandom` bằng ký hiệu `"*"`).

Các ca được chọn:

1. Example 1 của đề.
2. `remove` trên collection rỗng.
3. Thêm rồi xóa hết, rồi thêm lại (key phải được dọn sạch để `insert` lại trả `True`).
4. Toàn số trùng: 4 số `7`, xóa dần.
5. Giá trị biên `-2^31` và `2^31 - 1`, số âm.
6. Xóa phần tử cuối liên tục (tình huống B nhiều lần).

In [20]:
def expect(cls, ops, args, expected, seed=0):
    out = check_run(cls, ops, args, seed)
    for o, e in zip(out, expected):
        if e != "*":
            assert o == e, (ops, args, out, expected)
    return out


LO, HI = -2**31, 2**31 - 1
tests = [
    (ops1, args1, [None, True, False, True, "*", True, "*"]),
    (["RandomizedCollection", "remove", "insert", "remove", "remove"],
     [[], [0], [0], [0], [0]], [None, False, True, True, False]),
    (["RandomizedCollection", "insert", "insert", "remove", "remove", "insert", "getRandom"],
     [[], [4], [9], [4], [9], [4], []], [None, True, True, True, True, True, 4]),
    (["RandomizedCollection"] + ["insert"] * 4 + ["remove", "getRandom", "remove", "remove", "remove", "remove", "insert"],
     [[]] + [[7]] * 4 + [[7], [], [7], [7], [7], [7], [7]],
     [None, True, False, False, False, True, 7, True, True, True, False, True]),
    (["RandomizedCollection", "insert", "insert", "insert", "remove", "getRandom", "remove", "remove", "insert"],
     [[], [LO], [HI], [-1], [HI], [], [-1], [LO], [HI]],
     [None, True, True, True, True, "*", True, True, True]),
    (["RandomizedCollection", "insert", "insert", "insert", "remove", "remove", "remove", "insert"],
     [[], [1], [2], [3], [3], [2], [1], [2]],
     [None, True, True, True, True, True, True, True]),
]
for t, (ops, args, expected) in enumerate(tests, 1):
    print(f"test {t}:", show(expect(RandomizedCollection, ops, args, expected, seed=t)))
print("Tất cả test PASS!")

test 1: [null, true, false, true, 1, true, 2]
test 2: [null, false, true, true, false]
test 3: [null, true, true, true, true, true, 4]
test 4: [null, true, false, false, false, true, 7, true, true, true, false, true]
test 5: [null, true, true, true, true, -1, true, true, true]
test 6: [null, true, true, true, true, true, true, true]
Tất cả test PASS!


## 10. Cách 3b (nâng cao) — `list` chỉ số + "con trỏ ngược"

Ở mục 6.3, `list` chỉ số thua vì phải **dò tìm** `last_i` trong `pos[last]`. Nhưng nếu phần tử ở ô cuối **tự biết** nó nằm ở vị trí nào trong `pos[last]` thì khỏi phải dò!

Ý tưởng: mỗi ô của `nums` lưu một cặp `(v, k)` nghĩa là "ô này chứa giá trị `v`, và chỉ số của ô này được ghi ở `pos[v][k]`". Cặp `k` gọi là **con trỏ ngược** (back-pointer). Bất biến:

> `pos[v][k] == i`  **khi và chỉ khi**  `nums[i] == (v, k)`.

`remove(val)`:

1. Lấy chỉ số **cuối list** của `val`: `i = pos[val][-1]` (sắp `pop` nó ra, `O(1)`).
2. Phần tử ở ô cuối là `(lv, lk)`. Chuyển nó vào ô `i`: `nums[i] = (lv, lk)`, và nhờ con trỏ ngược, sửa thẳng `pos[lv][lk] = i` — **không cần dò**.
3. `pos[val].pop()` và `nums.pop()`.

Tình huống "xóa chính ô cuối" vẫn đúng: khi đó `(lv, lk)` chính là `(val, len(pos[val]) - 1)`, bước 2 ghi `pos[val][lk] = i` (không đổi gì), bước 3 bỏ cả hai.

Cách này **mọi thao tác trên list đều `O(1)` thật sự** (không có set), đổi lại code khó đọc hơn và dễ sai hơn. Với phỏng vấn, cách 3 (set) là đủ và phổ biến nhất; cách 3b đáng biết như một ví dụ đẹp về kỹ thuật con trỏ ngược. Ở đây `getRandom` chọn một cặp rồi trả `v`.

In [21]:
class RandomizedCollectionBackPtr:
    def __init__(self):
        self.nums = []        # mỗi ô là (v, k): pos[v][k] == chỉ số của ô này
        self.pos = {}         # v -> LIST các chỉ số trong nums

    def insert(self, val: int) -> bool:
        lst = self.pos.setdefault(val, [])     # setdefault: chưa có thì tạo [] rồi trả về
        lst.append(len(self.nums))
        self.nums.append((val, len(lst) - 1))  # con trỏ ngược tới vị trí trong lst
        return len(lst) == 1

    def remove(self, val: int) -> bool:
        if val not in self.pos:
            return False
        lst = self.pos[val]
        i = lst[-1]                            # xóa bản sao có chỉ số ghi ở cuối lst
        lv, lk = self.nums[-1]                 # phần tử ở ô cuối
        self.nums[i] = (lv, lk)                # chuyển nó vào ô i
        self.pos[lv][lk] = i                   # sửa địa chỉ nhờ con trỏ ngược: O(1)
        lst.pop()
        self.nums.pop()
        if not lst:
            del self.pos[val]
        return True

    def getRandom(self) -> int:
        return random.choice(self.nums)[0]


def backptr_consistent(obj):
    return all(obj.pos[v][k] == i for i, (v, k) in enumerate(obj.nums)) and \
        sum(len(l) for l in obj.pos.values()) == len(obj.nums)


for t, (ops, args, expected) in enumerate(tests, 1):
    expect(RandomizedCollectionBackPtr, ops, args, expected, seed=t)

bp = RandomizedCollectionBackPtr()
for op, (val,) in zip(trace_ops, trace_args):
    getattr(bp, op)(val)
print("nums =", bp.nums)
print("pos  =", bp.pos, "| nhất quán:", backptr_consistent(bp))
print("Cách 3b PASS toàn bộ test của mục 9.1")

nums = [(1, 0), (2, 0), (2, 1)]
pos  = {1: [0], 2: [1, 2]} | nhất quán: True
Cách 3b PASS toàn bộ test của mục 9.1


## 11. Cách 4 — Kiểm chứng phân phối của `getRandom` bằng thực nghiệm

Các test ở trên chỉ kiểm tra `getRandom` trả về giá trị **có mặt**. Nhưng một `getRandom` luôn trả `nums[0]` cũng qua được kiểm tra đó, dù hoàn toàn không ngẫu nhiên! LeetCode kiểm tra **phân phối**, nên ta cũng phải tự kiểm tra.

Phương pháp thực nghiệm:

1. Dựng một collection cụ thể, biết trước tần suất lý thuyết `c(v) / n`.
2. Gọi `getRandom` thật nhiều lần (`TRIALS = 60 000`), đếm bằng `Counter`.
3. So tần suất thực nghiệm với lý thuyết; độ lệch lớn nhất phải nhỏ (ở đây yêu cầu `< 0.01`).

Vì sao `0.01` là ngưỡng hợp lý? Với `N` lần thử, độ lệch chuẩn của tần suất một giá trị có xác suất `p` là `sqrt(p(1-p)/N)`. Với `p = 0.5`, `N = 60 000`: `sqrt(0.25 / 60000) ≈ 0.002`. Ngưỡng `0.01` gấp khoảng 5 lần → một cài đặt đúng gần như không bao giờ vượt, còn một cài đặt sai phân phối (lệch cỡ `0.1`) bị bắt ngay. Ta còn đặt `random.seed` để kết quả lặp lại được.

Ta kiểm tra 3 trạng thái: `[1, 1, 2]` (Example 1), sau `remove(1)`, và một collection lớn hơn sau nhiều lần thêm / xóa.

In [22]:
def empirical(obj, trials=60_000):
    freq = Counter(obj.getRandom() for _ in range(trials))
    return {v: freq[v] / trials for v in sorted(freq)}


def compare_distribution(name, obj, multiset, trials=60_000):
    theory = {v: c / len(multiset) for v, c in sorted(Counter(multiset).items())}
    emp = empirical(obj, trials)
    worst = max(abs(emp.get(v, 0) - p) for v, p in theory.items())
    assert set(emp) <= set(theory), "getRandom trả giá trị lạ"
    assert worst < 0.01, (name, emp, theory)
    print(f"{name:<22} lý thuyết={ {v: round(p, 3) for v, p in theory.items()} }")
    print(f"{'':<22} thực nghiệm={ {v: round(p, 3) for v, p in emp.items()} } | lệch max={worst:.4f}")


random.seed(2026)
rc = RandomizedCollection()
for v in [1, 1, 2]:
    rc.insert(v)
compare_distribution("[1, 1, 2]", rc, [1, 1, 2])
rc.remove(1)
compare_distribution("sau remove(1)", rc, [1, 2])

rc = RandomizedCollection()
for v in [5] * 3 + [7] + [9] * 6:
    rc.insert(v)
rc.remove(9)
rc.remove(9)
rc.remove(4)                                   # không có: không đổi gì
compare_distribution("5x3, 7x1, 9x4", rc, [5] * 3 + [7] + [9] * 4)

[1, 1, 2]              lý thuyết={1: 0.667, 2: 0.333}
                       thực nghiệm={1: 0.672, 2: 0.328} | lệch max=0.0051


sau remove(1)          lý thuyết={1: 0.5, 2: 0.5}
                       thực nghiệm={1: 0.498, 2: 0.502} | lệch max=0.0018


5x3, 7x1, 9x4          lý thuyết={5: 0.375, 7: 0.125, 9: 0.5}
                       thực nghiệm={5: 0.376, 7: 0.123, 9: 0.501} | lệch max=0.0021


### 11.1 Thực nghiệm bắt được một `getRandom` sai

Lỗi phổ biến: viết `getRandom` chọn đều **trên các giá trị khác nhau** (`random.choice(list(self.pos))`). Code này trả về giá trị có mặt, nên `check_run` không phát hiện. Nhưng với `[1, 1, 2]` nó cho `1` và `2` mỗi giá trị khoảng `1/2` thay vì `2/3` và `1/3` — sai đề. (Ngoài ra `list(self.pos)` tốn `O(k)`, cũng vi phạm yêu cầu `O(1)`.)

Cell dưới chạy cả hai phiên bản trên cùng collection để so sánh.

In [23]:
class WrongRandomByKey(RandomizedCollection):
    def getRandom(self) -> int:
        return random.choice(list(self.pos))   # SAI: đều theo giá trị, bỏ qua số lần xuất hiện


random.seed(7)
for cls in [RandomizedCollection, WrongRandomByKey]:
    obj = cls()
    for v in [1, 1, 2]:
        obj.insert(v)
    emp = empirical(obj)
    verdict = "ĐÚNG" if abs(emp[1] - 2 / 3) < 0.01 else "SAI phân phối"
    print(f"{cls.__name__:<22} P(1)≈{emp[1]:.3f}, P(2)≈{emp[2]:.3f}  (cần 0.667 / 0.333) -> {verdict}")

# check_run vẫn cho qua bản sai -> chứng tỏ cần thêm kiểm tra phân phối
check_run(WrongRandomByKey, ops1, args1, seed=0)
print("check_run không phát hiện được bản sai!")

RandomizedCollection   P(1)≈0.666, P(2)≈0.334  (cần 0.667 / 0.333) -> ĐÚNG


WrongRandomByKey       P(1)≈0.502, P(2)≈0.498  (cần 0.667 / 0.333) -> SAI phân phối
check_run không phát hiện được bản sai!


## 12. Vì sao cách 3 luôn đúng? Độ phức tạp

### 12.1 Ba bất biến (invariant)

Sau **mỗi** lời gọi `insert` / `remove`, ba điều sau luôn đúng:

- **(I1)** `nums` chứa đúng các phần tử của multiset, mỗi bản sao một ô (thứ tự tùy ý).
- **(I2)** Với mọi giá trị `v`: `pos[v]` = **đúng** tập các chỉ số `i` có `nums[i] == v` — không thiếu, không thừa.
- **(I3)** Không có key nào ứng với set rỗng; tức là `v in pos` ⇔ `v` đang có mặt.

**Ban đầu:** `nums = []`, `pos` rỗng → cả ba đúng.

**`insert(val)`:** ô mới có chỉ số `n` chứa `val`; ta thêm `n` vào `pos[val]` → (I1), (I2) giữ. `pos[val]` vừa được thêm nên khác rỗng → (I3) giữ. Theo (I2), trước khi thêm `val` có mặt ⇔ `pos[val]` khác rỗng ⇔ sau khi thêm `len(pos[val]) >= 2`. Vậy `len == 1` ⇔ trước đó chưa có → giá trị trả về đúng.

**`remove(val)`:** theo (I3), `val not in pos` ⇔ không có mặt → trả `False` đúng. Ngược lại, gọi `i` là chỉ số lấy ra, `L = n - 1`, `last = nums[L]`. Sau khi chạy, ô `i` chứa `last`, ô `L` biến mất, các ô khác giữ nguyên. Ta kiểm tra (I2) cho từng tình huống:

- **A** (`i != L`, `last != val`): `pos[val]` mất `i` (ô `i` không còn chứa `val`). `pos[last]` thêm `i`, bỏ `L` (ô `i` giờ chứa `last`, ô `L` biến mất). Đúng.
- **B** (`i == L`): ô `L` chứa `val` bị bỏ. `pos[val]` mất `L` ở bước `pop`; `add(L)` rồi `discard(L)` triệt tiêu nhau. Kết quả `pos[val]` mất đúng `L`. Đúng.
- **C** (`i != L`, `last == val`): ô `i` vẫn chứa `val`, ô `L` biến mất. `pos[val]`: bỏ `i`, thêm lại `i`, bỏ `L` → mất đúng `L`. Đúng.

Cuối cùng dòng `if not self.pos[val]: del ...` giữ (I3); `pos[last]` với `last != val` luôn còn ít nhất `i` nên không rỗng.

**`getRandom`:** theo (I1), `nums` chứa đúng `c(v)` ô mang giá trị `v`; `random.choice` chọn đều một trong `n` ô → `P(v) = c(v) / n`. Đúng yêu cầu đề (mục 2.4).

Cell dưới kiểm tra (I1), (I2), (I3) **sau từng lời gọi** trên hàng trăm chuỗi lệnh ngẫu nhiên có rất nhiều số trùng (giá trị chỉ trong `0..3`).

In [24]:
def check_invariants(obj, oracle):
    assert Counter(obj.nums) == oracle, "I1 sai"                         # đúng multiset
    for v, s in obj.pos.items():
        assert s == {i for i, x in enumerate(obj.nums) if x == v}, "I2 sai"
        assert s, "I3 sai: set rỗng"
    assert set(obj.pos) == set(oracle), "I2/I3 sai: thiếu hoặc thừa key"


random.seed(381)
calls = 0
for _ in range(300):
    obj, oracle = RandomizedCollection(), Counter()
    for _ in range(random.randint(1, 40)):
        val = random.randint(0, 3)
        if random.random() < 0.55:
            assert obj.insert(val) == (oracle[val] == 0)
            oracle[val] += 1
        else:
            present = oracle[val] > 0
            assert obj.remove(val) == present
            if present:
                oracle[val] -= 1
        oracle = +oracle                      # +Counter: bỏ các key có số đếm 0
        check_invariants(obj, oracle)
        calls += 1
print(f"Đã kiểm tra 3 bất biến sau {calls} lời gọi ngẫu nhiên: PASS")

Đã kiểm tra 3 bất biến sau 5916 lời gọi ngẫu nhiên: PASS


### 12.2 Độ phức tạp cách 3

| Thao tác | Thời gian | Lý do |
|---|---|---|
| `insert` | `O(1)` trung bình | 1 lần `set.add` + 1 lần `list.append` (khấu hao `O(1)`) + truy cập `dict` |
| `remove` | `O(1)` trung bình | vài thao tác `dict` / `set` (`pop`, `add`, `discard`) + gán `nums[i]` + `list.pop()` |
| `getRandom` | `O(1)` | `random.choice` chọn một chỉ số |

- **Bộ nhớ:** `O(n)` — `nums` có `n` ô, và tổng kích thước mọi `pos[v]` cũng đúng bằng `n` (mỗi ô được ghi đúng một lần, theo (I2)).
- **"Trung bình"** vì `dict` / `set` dùng hash: thường `O(1)`, nhưng về lý thuyết nếu rất nhiều key va chạm hash thì có thể chậm hơn. Với số nguyên trong Python, điều này không xảy ra trong thực tế. Đề cũng chỉ đòi "`O(1)` trung bình".
- **Tổng cho `q` lời gọi:** `O(q)`; với `q = 2·10^5` là vài trăm nghìn thao tác — rất nhanh.
- Cách 3b có cùng độ phức tạp, nhưng phần "tập chỉ số" chỉ dùng thao tác `list` ở cuối (`append` / `pop`) và gán theo chỉ số, nên không phụ thuộc hash của `set`.

## 13. Test đối chiếu chéo và đo thời gian

### 13.1 Đối chiếu tất cả các cách trên chuỗi lệnh ngẫu nhiên

Sinh 400 chuỗi lệnh ngẫu nhiên (giá trị trong `-3..3` để có nhiều số trùng và nhiều lần `remove` giá trị không có; `getRandom` chỉ gọi khi collection khác rỗng, đúng như đề hứa). Với mỗi chuỗi, chạy **cả 5 cài đặt**: cách 1 (list), cách 2 (dict đếm), cách 3 (bản demo và bản LeetCode), cách 3b (con trỏ ngược). Yêu cầu:

- Mọi kết quả `insert` / `remove` của 5 cách **giống hệt nhau** và đúng theo "trọng tài" `Counter` (nhờ `check_run`).
- Mọi kết quả `getRandom` thuộc collection lúc đó.

In [25]:
def random_commands(length, lo=-3, hi=3):
    # Sinh chuỗi lệnh ngẫu nhiên; bag mô phỏng collection để chỉ gọi getRandom khi khác rỗng
    ops, args, bag = ["RandomizedCollection"], [[]], []
    for _ in range(length):
        r = random.random()
        if bag and r < 0.2:
            ops.append("getRandom"); args.append([])
        elif r < 0.6:
            v = random.randint(lo, hi)
            ops.append("insert"); args.append([v]); bag.append(v)
        else:
            v = random.randint(lo, hi)
            ops.append("remove"); args.append([v])
            if v in bag:
                bag.remove(v)
    return ops, args


ALL = [RandomizedCollectionList, RandomizedCollectionCounter, RandomizedCollectionSets,
       RandomizedCollection, RandomizedCollectionBackPtr]
random.seed(12345)
cases = [random_commands(random.randint(1, 30)) for _ in range(400)]
for t, (ops, args) in enumerate(cases):
    results = []
    for cls in ALL:
        out = check_run(cls, ops, args, seed=t)
        results.append([o for op, o in zip(ops, out) if op in ("insert", "remove")])
    assert all(r == results[0] for r in results), (ops, args)
print(f"{len(cases)} chuỗi lệnh ngẫu nhiên: 5 cách cho kết quả insert/remove giống hệt nhau, getRandom hợp lệ.")

400 chuỗi lệnh ngẫu nhiên: 5 cách cho kết quả insert/remove giống hệt nhau, getRandom hợp lệ.


### 13.2 Đo thời gian: thấy được `O(n)` và `O(1)`

Kịch bản "xấu" cho cả cách 1 và cách 2: thêm `n` số **khác nhau**, gọi `getRandom` `n` lần, rồi xóa từng số theo đúng thứ tự đã thêm (luôn xóa ở **đầu** list — tệ nhất cho cách 1).

- Cách 1: `insert` và `remove` mỗi lần `O(n)` → tổng `O(n^2)`.
- Cách 2: `getRandom` mỗi lần `O(k) = O(n)` → tổng `O(n^2)`.
- Cách 3, 3b: mọi thao tác `O(1)` → tổng `O(n)`.

Khi `n` gấp đôi, thời gian của `O(n^2)` tăng khoảng **4 lần**, còn `O(n)` tăng khoảng **2 lần**. Con số tuyệt đối tùy máy, hãy nhìn tỉ lệ.

In [26]:
def workload(cls, n):
    obj = cls()
    for v in range(n):
        obj.insert(v)
    for _ in range(n):
        obj.getRandom()
    for v in range(n):
        obj.remove(v)


random.seed(0)
print(f"{'n':>6} | {'Cách 1 list':>12} | {'Cách 2 đếm':>12} | {'Cách 3 set':>12} | {'Cách 3b':>12}")
for n in [2000, 4000, 8000]:
    times = [time_it(lambda c=cls: workload(c, n)) for cls in
             [RandomizedCollectionList, RandomizedCollectionCounter, RandomizedCollection, RandomizedCollectionBackPtr]]
    print(f"{n:>6} | " + " | ".join(f"{t:>11.4f}s" for t in times))

     n |  Cách 1 list |   Cách 2 đếm |   Cách 3 set |      Cách 3b


  2000 |      0.0106s |      0.0571s |      0.0015s |      0.0012s


  4000 |      0.0410s |      0.2258s |      0.0031s |      0.0023s


  8000 |      0.1669s |      0.9306s |      0.0078s |      0.0048s


### 13.3 Bảng so sánh các cách

| Cách | Cấu trúc | `insert` | `remove` | `getRandom` | Bộ nhớ | Nhận xét |
|---|---|---|---|---|---|---|
| 1. Chỉ `list` | `list` mọi bản sao | `O(n)` | `O(n)` | `O(1)` | `O(n)` | đúng, dễ viết; làm baseline; TLE |
| 2. `dict` đếm | `dict` giá trị → số lần | `O(1)` | `O(1)` | `O(k)` | `O(k)` | phân phối đúng nhờ "vị trí ảo"; TLE khi nhiều giá trị khác nhau |
| **3. `list` + `dict` → `set` chỉ số** | `list` + `defaultdict(set)` | **`O(1)`** tb | **`O(1)`** tb | **`O(1)`** | `O(n)` | **cách chính**; nhớ "add trước, discard sau" |
| 3b. `list` + `list` chỉ số + con trỏ ngược | `list[(v, k)]` + `dict` → `list` | `O(1)` tb | `O(1)` tb | `O(1)` | `O(n)` | không cần `set`; khó đọc hơn |
| 4. Kiểm chứng phân phối | `Counter` trên nhiều lần gọi | | | | | không phải lời giải, mà là **công cụ test** bắt lỗi phân phối |

(`tb` = trung bình; `n` = số phần tử đang có, `k` = số giá trị khác nhau.)

## 14. Các lỗi người mới hay gặp

### Lỗi 1: `discard` trước `add` (đảo thứ tự hai dòng)

```python
self.pos[last].discard(last_i)   # SAI thứ tự
self.pos[last].add(i)
```

Chạy với tình huống B đơn giản nhất: `insert(5)` rồi `remove(5)`. `pop()` lấy `i = 0 = last_i`; `discard(0)` không có gì để bỏ; `add(0)` **thêm lại** `0` → `pos = {5: {0}}` trong khi `nums = []`. Hậu quả dây chuyền: `insert(5)` sau đó trả `False` (sai: collection đang rỗng), và `remove(5)` lần nữa trả `True` hoặc làm sập chương trình vì truy cập ô không tồn tại.

In [27]:
class BugDiscardFirst(RandomizedCollection):
    def remove(self, val: int) -> bool:
        if val not in self.pos:
            return False
        i = self.pos[val].pop()
        last_i = len(self.nums) - 1
        last = self.nums[last_i]
        self.nums[i] = last
        self.pos[last].discard(last_i)      # SAI: discard trước ...
        self.pos[last].add(i)               # ... add sau -> "hồi sinh" chỉ số ma
        self.nums.pop()
        if not self.pos[val]:
            del self.pos[val]
        return True


bug = BugDiscardFirst()
bug.insert(5)
print("remove(5):", bug.remove(5), "| nums =", bug.nums, "| pos =", dict(bug.pos), "<- chỉ số 0 'ma'")
try:
    print("remove(5) lần 2:", bug.remove(5))
except IndexError as e:
    print("remove(5) lần 2 -> IndexError:", e)

ok = RandomizedCollection()
ok.insert(5)
print("Bản đúng: remove(5):", ok.remove(5), "| nums =", ok.nums, "| pos =", dict(ok.pos),
      "| remove(5) lần 2:", ok.remove(5))

remove(5): True | nums = [] | pos = {5: {0}} <- chỉ số 0 'ma'
remove(5) lần 2 -> IndexError: list index out of range
Bản đúng: remove(5): True | nums = [] | pos = {} | remove(5) lần 2: False


### Lỗi 2: Xóa ô `i` bằng `del self.nums[i]` thay vì đổi chỗ

`del self.nums[i]` (hay `self.nums.pop(i)`) không chỉ tốn `O(n)`, mà còn làm mọi phần tử phía sau **dời chỉ số** đi 1, trong khi `pos` vẫn nhớ chỉ số cũ → `pos` sai hàng loạt.

### Lỗi 3: Không xóa key khi set rỗng, nhưng lại kiểm tra bằng `val in self.pos`

Sau `insert(1)`, `remove(1)` ta có `pos = {1: set()}`. Nếu không dọn key, lần `remove(1)` kế tiếp thấy `1 in pos` là `True`, gọi `set().pop()` → `KeyError: 'pop from an empty set'`. Có hai cách sửa: luôn `del` key khi rỗng (như lời giải), **hoặc** kiểm tra bằng `if not self.pos[val]` (nhưng nhớ `defaultdict` sẽ tạo key rỗng mỗi lần kiểm tra giá trị chưa từng có — vô hại về đúng sai nhưng tốn bộ nhớ).

Cell dưới chạy thử cả hai lỗi.

In [28]:
class BugDelShift(RandomizedCollection):
    def remove(self, val: int) -> bool:
        if val not in self.pos:
            return False
        i = self.pos[val].pop()
        del self.nums[i]                    # SAI: các phần tử sau i bị dời chỉ số
        if not self.pos[val]:
            del self.pos[val]
        return True


b2 = BugDelShift()
for v in [1, 2, 3]:
    b2.insert(v)
b2.remove(1)
print("Lỗi 2: nums =", b2.nums, "| pos =", dict(b2.pos), "| nhất quán:", consistent(b2.nums, b2.pos))
try:
    b2.remove(3)                            # pos[3] = {2} nhưng ô 2 không còn
    print("remove(3) xong nhưng nums =", b2.nums)
except IndexError as e:
    print("remove(3) -> IndexError:", e)


class BugNoCleanup(RandomizedCollection):
    def remove(self, val: int) -> bool:
        if val not in self.pos:             # dựa vào 'in' ...
            return False
        i = self.pos[val].pop()
        last_i = len(self.nums) - 1
        last = self.nums[last_i]
        self.nums[i] = last
        self.pos[last].add(i)
        self.pos[last].discard(last_i)
        self.nums.pop()
        return True                         # ... nhưng QUÊN xóa key khi set rỗng


b3 = BugNoCleanup()
b3.insert(1)
print("Lỗi 3: remove(1):", b3.remove(1), "| pos =", dict(b3.pos))
try:
    b3.remove(1)
except KeyError as e:
    print("remove(1) lần 2 -> KeyError:", e)

Lỗi 2: nums = [2, 3] | pos = {2: {1}, 3: {2}} | nhất quán: False
remove(3) -> IndexError: list assignment index out of range
Lỗi 3: remove(1): True | pos = {1: set()}
remove(1) lần 2 -> KeyError: 'pop from an empty set'


### Lỗi 4: `insert` trả về sai

Hai cách viết sai hay gặp:

```python
self.pos[val].add(len(self.nums)); self.nums.append(val)
return val not in self.pos          # SAI: vừa thêm xong thì val luôn có mặt -> luôn False
```

```python
return len(self.pos[val]) == 0      # SAI: kiểm tra SAU khi thêm thì không bao giờ bằng 0
```

Sửa: kiểm tra **sau** khi thêm thì dùng `len(self.pos[val]) == 1`; hoặc lưu `was_new = val not in self.pos` **trước** khi thêm rồi trả `was_new`.

### Lỗi 5: `getRandom` chọn đều theo giá trị khác nhau

`random.choice(list(self.pos))` hoặc `random.choice(list(set(self.nums)))`: vừa sai phân phối (mục 11.1), vừa `O(k)` / `O(n)`. Phải chọn trên `self.nums` — list chứa **mọi bản sao**.

### Lỗi 6: Dùng `list` cho tập chỉ số rồi gọi `.remove(last_i)`

Kết quả vẫn đúng, nhưng `remove` thành `O(c)`; khi toàn số trùng (mục 6.3) thì tổng thành `O(n^2)` → TLE. Dùng `set`, hoặc kỹ thuật con trỏ ngược (mục 10).

### Lỗi 7: `random.choice` trên `set` / `dict`; khởi tạo set bằng `{}`

`random.choice(self.pos[val])` → `TypeError` (set không có chỉ số). `x = {}` tạo **`dict`** rỗng chứ không phải set → `x.add(...)` báo `AttributeError`. Dùng `set()` hoặc `defaultdict(set)`.

### Bảng tóm tắt lỗi

| Lỗi | Triệu chứng | Sửa |
|---|---|---|
| 1. `discard` trước `add` | chỉ số "ma", `insert` sau đó trả sai, `IndexError` | `add(i)` **trước**, `discard(last_i)` **sau** |
| 2. `del nums[i]` | `pos` lệch chỉ số, `IndexError`, `O(n)` | đổi chỗ với phần tử cuối rồi `pop()` |
| 3. Không dọn key rỗng | `KeyError: pop from an empty set`, `remove` trả `True` sai | `if not self.pos[val]: del self.pos[val]` |
| 4. `insert` trả sai | luôn `False` / luôn `True` | `len(self.pos[val]) == 1` sau khi thêm |
| 5. `getRandom` theo key | phân phối đều theo giá trị, sai đề | `random.choice(self.nums)` |
| 6. `list` chỉ số + `.remove` | đúng nhưng TLE khi nhiều số trùng | dùng `set` (hoặc con trỏ ngược) |
| 7. `choice(set)`, `{}` | `TypeError`, `AttributeError` | chọn trên list; `set()` / `defaultdict(set)` |

## 15. Mẹo nhận dạng pattern, biến thể và bài liên quan

### Dấu hiệu nhận dạng

- Đề đòi **"chọn ngẫu nhiên đều"** + **thêm / xóa** đều `O(1)` → nghĩ ngay tới **mảng + hash map giá trị → chỉ số** và mẹo **đổi chỗ với phần tử cuối rồi `pop`**.
- Cho phép **trùng** → hash map giá trị → **tập** chỉ số (`set`), hoặc list chỉ số + con trỏ ngược.
- "Xác suất tỉ lệ với số lần xuất hiện" → lưu **đủ mọi bản sao** trong mảng rồi chọn đều một ô.
- "Thứ tự không quan trọng" + "xóa ở giữa mảng" → **swap-with-last + pop** (mẹo này còn dùng trong game để xóa nhân vật khỏi danh sách, trong scheduler để bỏ task...).

### Template trong đầu

```text
nums : list mọi bản sao           pos : giá trị -> set chỉ số
insert(v):  pos[v].add(len(nums)); nums.append(v); return len(pos[v]) == 1
remove(v):  không có -> False
            i = pos[v].pop(); L = len(nums)-1; last = nums[L]
            nums[i] = last; pos[last].add(i); pos[last].discard(L)   # add trước, discard sau
            nums.pop(); pos[v] rỗng -> del
getRandom:  random.choice(nums)
```

### Biến thể và bài liên quan

| Bài | Liên hệ |
|---|---|
| 380. Insert Delete GetRandom O(1) (Medium) | bản không trùng: `dict` giá trị → **một** chỉ số (mục 3, Bài tập 4) |
| 528. Random Pick with Weight (Medium) | xác suất tỉ lệ với **trọng số**: tổng tiền tố + tìm kiếm nhị phân — chính là ý "vị trí ảo" của cách 2, làm nhanh hơn (Bài tập 6) |
| 398. Random Pick Index (Medium) | chọn ngẫu nhiên một chỉ số của `target` trong mảng có trùng: `dict` giá trị → list chỉ số, hoặc reservoir sampling |
| 382. Linked List Random Node (Medium) | chọn đều khi **không biết trước độ dài**: reservoir sampling |
| 710. Random Pick with Blacklist (Hard) | ánh xạ lại các chỉ số "bị cấm" vào phần cuối mảng — họ hàng của mẹo đổi chỗ với cuối |
| 146. LRU Cache (Medium) | cũng là bài thiết kế kết hợp **hai** cấu trúc (hash map + danh sách liên kết) để mọi thao tác `O(1)` |

## 16. Bài tập vận dụng

Làm theo thứ tự, mỗi bài khó hơn bài trước. Gợi ý ở mục 17, đáp án ở mục 18 (cuối notebook) — hãy tự làm trước!

### Bài 1 — Đoán output (không chạy code)

```python
a = [4, 8, 15, 16, 23]
i = 1
a[i] = a[-1]
a.pop()
print(a)

s = {3, 5}
s.add(5)
s.discard(9)
print(len(s), 9 in s)

pos = defaultdict(set)
print(len(pos), 7 in pos, pos[7], 7 in pos)

print(Counter([2, 2, 2, 6])[2] / 4)
```

Viết ra 4 dòng output và giải thích. Câu hỏi thêm: với collection `[2, 2, 2, 6]`, `getRandom` phải trả `6` với xác suất bao nhiêu?

### Bài 2 — Tính tay bảng trạng thái

Chạy cách 3 trên chuỗi lệnh `insert 4, insert 4, insert 7, insert 4, remove 4, remove 7, remove 4, remove 4, remove 4`.

Để bài có đáp án duy nhất, **giả sử** mỗi lần `remove`, chỉ số được lấy ra khỏi `pos[val]` là chỉ số **nhỏ nhất** trong set (thay cho `set.pop()`). Lập bảng giống mục 8.2: `i`, `last_i`, `last`, tình huống A/B/C, `nums`, `pos`, giá trị trả về. Lệnh nào trả `False`?

### Bài 3 — Điền TODO: `remove` của cách 3

Hoàn thiện `remove`. Nhớ ba tình huống A / B / C và quy tắc "add trước, discard sau".

In [29]:
class RandomizedCollectionExercise:
    def __init__(self):
        self.nums = []
        self.pos = defaultdict(set)

    def insert(self, val: int) -> bool:
        self.pos[val].add(len(self.nums))
        self.nums.append(val)
        return len(self.pos[val]) == 1

    def remove(self, val: int) -> bool:
        # TODO 1: val không có mặt -> return False
        # TODO 2: lấy ra một chỉ số i của val; tính last_i và last
        # TODO 3: chuyển last vào ô i; cập nhật pos[last] (thứ tự nào?)
        # TODO 4: bỏ ô cuối; dọn key nếu pos[val] rỗng; return True
        pass

    def getRandom(self) -> int:
        return random.choice(self.nums)


# Tự làm xong rồi mới mở test:
# for t, (ops, args, expected) in enumerate(tests, 1):
#     expect(RandomizedCollectionExercise, ops, args, expected, seed=t)

### Bài 4 — Tự viết lại: LeetCode 380 `RandomizedSet`

Không nhìn lại mục 3, tự viết `RandomizedSet` (không cho trùng): `insert` trả `False` và **không thêm** nếu đã có; `remove` trả `False` nếu không có; `getRandom` đều trên các phần tử. Cả ba `O(1)` trung bình.

```text
Input
["RandomizedSet", "insert", "remove", "insert", "getRandom", "remove", "insert", "getRandom"]
[[], [1], [2], [2], [], [1], [2], []]
Output
[null, true, false, true, 2, true, false, 2]
```

In [30]:
class RandomizedSetExercise:
    def __init__(self):
        # TODO: list giá trị + dict giá trị -> chỉ số
        raise NotImplementedError

    def insert(self, val: int) -> bool:
        raise NotImplementedError

    def remove(self, val: int) -> bool:
        # TODO: đổi chỗ với phần tử cuối; cập nhật trước, xóa sau
        raise NotImplementedError

    def getRandom(self) -> int:
        raise NotImplementedError


# Gợi ý test:
# random.seed(0)
# out = run_commands(RandomizedSetExercise, ops380, args380)
# assert [o for o, op in zip(out, ops380) if op != "getRandom"] == [None, True, False, True, True, False]

### Bài 5 — Biến thể: thêm `count(val)` và `getRandomDistinct()`

Mở rộng cách 3 thành class `RandomizedCollectionPlus` có thêm:

- `count(val) -> int`: số lần `val` xuất hiện, `O(1)`.
- `getRandomDistinct() -> int`: trả một giá trị ngẫu nhiên, **đều trên các giá trị khác nhau** (với `[1, 1, 2]`: `1` và `2` mỗi giá trị xác suất `1/2`), cũng phải `O(1)` trung bình.

Chú ý: `random.choice(list(self.pos))` là `O(k)` → không đạt. Hãy nghĩ: tập các **giá trị khác nhau** cũng là một "bài 380" thu nhỏ — giá trị mới xuất hiện thì thêm, giá trị hết bản sao thì xóa.

In [31]:
class RandomizedCollectionPlusExercise(RandomizedCollection):
    def __init__(self):
        super().__init__()
        # TODO: thêm cấu trúc cho tập giá trị khác nhau (list + dict giá trị -> chỉ số)

    def insert(self, val: int) -> bool:
        # TODO: gọi super().insert(val); nếu là giá trị mới thì thêm vào tập giá trị khác nhau
        raise NotImplementedError

    def remove(self, val: int) -> bool:
        # TODO: gọi super().remove(val); nếu val hết bản sao thì xóa khỏi tập giá trị khác nhau (O(1))
        raise NotImplementedError

    def count(self, val: int) -> int:
        raise NotImplementedError

    def getRandomDistinct(self) -> int:
        raise NotImplementedError


# Gợi ý test:
# obj = RandomizedCollectionPlusExercise()
# for v in [1, 1, 2]: obj.insert(v)
# assert obj.count(1) == 2 and obj.count(9) == 0
# freq = Counter(obj.getRandomDistinct() for _ in range(60000))
# assert abs(freq[1] / 60000 - 0.5) < 0.01

### Bài 6 — Mini challenge: LeetCode 528. Random Pick with Weight

Cho mảng trọng số dương `w` (chỉ số từ `0`). Cài đặt class `Solution`:

- `Solution(w)` khởi tạo với `w`.
- `pickIndex() -> int` trả về một chỉ số `i` trong `[0, len(w) - 1]` với xác suất `w[i] / sum(w)`.

Ví dụ `w = [1, 3]`: trả `0` với xác suất `1/4`, trả `1` với xác suất `3/4`.

Đây chính là ý "vị trí ảo" của cách 2: giá trị `i` chiếm `w[i]` vị trí liên tiếp. Nhưng thay vì dò tuần tự `O(n)`, hãy làm `pickIndex` trong `O(log n)`. Kiểm chứng phân phối bằng `Counter` như mục 11. (Ràng buộc đề gốc: `1 <= w.length <= 10^4`, `1 <= w[i] <= 10^5`, `pickIndex` được gọi tối đa `10^4` lần.)

In [32]:
from typing import List


class WeightedPickExercise:
    def __init__(self, w: List[int]):
        # TODO: tính mảng tổng tiền tố prefix, prefix[i] = w[0] + ... + w[i]
        pass

    def pickIndex(self) -> int:
        # TODO: chọn r ngẫu nhiên trong [0, tổng); tìm chỉ số i NHỎ NHẤT có prefix[i] > r
        pass


# Gợi ý test:
# random.seed(1)
# wp = WeightedPickExercise([1, 3])
# freq = Counter(wp.pickIndex() for _ in range(60000))
# assert abs(freq[0] / 60000 - 0.25) < 0.01 and abs(freq[1] / 60000 - 0.75) < 0.01

## 17. Gợi ý theo mức

Chỉ mở mức tiếp theo khi mức trước chưa đủ.

| Bài | Mức 1 — hướng tư duy | Mức 2 — cấu trúc / cú pháp | Mức 3 — gần code |
|---|---|---|---|
| 1 | Swap-with-last: phần tử cuối nhảy vào ô `i`. | `set.add` phần tử đã có không đổi gì; `discard` phần tử không có thì im lặng. | `defaultdict`: `in` không tạo key, nhưng **đọc** `pos[7]` thì tạo. |
| 2 | Mỗi `remove`: lấy `i` nhỏ nhất → xác định `last_i`, `last` → A/B/C. | Lần `remove 4` đầu: `pos[4] = {0, 1, 3}`, `i = 0`, `last_i = 3`, `last = 4` → C. | Khi collection rỗng mà còn `remove 4` → `False`. |
| 3 | Đúng 5 bước như mục 7.2. | `i = self.pos[val].pop()`; `last_i = len(self.nums) - 1`. | `self.pos[last].add(i)` rồi mới `self.pos[last].discard(last_i)`. |
| 4 | `dict` giá trị → **một** chỉ số. | `remove`: `i = idx[val]`, `last = nums[-1]`, `nums[i] = last`. | `idx[last] = i`, `nums.pop()`, `del idx[val]` (đúng thứ tự này). |
| 5 | Tập giá trị khác nhau = một `RandomizedSet` bên trong. | `count(val)` = `len(self.pos[val])` nếu `val in self.pos`, không thì `0`. | `insert` trả `True` → thêm `val` vào tập khác nhau; sau `remove` mà `val not in self.pos` → xóa `val` khỏi tập đó. |
| 6 | Trọng số `w[i]` = số "vị trí ảo" của chỉ số `i`. | `prefix = list(itertools.accumulate(w))`; `r = random.randrange(prefix[-1])`. | `bisect.bisect_right(prefix, r)` trả chỉ số nhỏ nhất có `prefix[i] > r` trong `O(log n)`. |

## 18. Đáp án tham khảo

Nên tự làm trước rồi mới xem phần này.

### Đáp án Bài 1

```text
[4, 23, 15, 16]
2 False
0 False set() True
0.75
```

- `a[1] = a[-1]` chép `23` đè lên `8` → `[4, 23, 15, 16, 23]`; `pop()` bỏ ô cuối → `[4, 23, 15, 16]`.
- `s.add(5)`: `5` đã có → không đổi. `discard(9)`: không có → im lặng. Còn `{3, 5}` → `2 False`.
- `defaultdict` rỗng: `len = 0`, `7 in pos` là `False` (không tạo key); **đọc** `pos[7]` tạo ra `set()` rỗng; sau đó `7 in pos` là `True`.
- `Counter([2, 2, 2, 6])[2] = 3` → `3 / 4 = 0.75`. Câu hỏi thêm: `6` xuất hiện 1 trong 4 ô → xác suất `1/4 = 0.25`.

### Đáp án Bài 2

| # | Lệnh | `i` | `last_i` | `last` | Tình huống | `nums` sau | `pos` sau | Trả về |
|---:|---|---:|---:|---:|---|---|---|---|
| 1 | `insert(4)` | | | | | `[4]` | `{4:{0}}` | `True` |
| 2 | `insert(4)` | | | | | `[4, 4]` | `{4:{0,1}}` | `False` |
| 3 | `insert(7)` | | | | | `[4, 4, 7]` | `{4:{0,1}, 7:{2}}` | `True` |
| 4 | `insert(4)` | | | | | `[4, 4, 7, 4]` | `{4:{0,1,3}, 7:{2}}` | `False` |
| 5 | `remove(4)` | 0 | 3 | 4 | C | `[4, 4, 7]` | `{4:{0,1}, 7:{2}}` | `True` |
| 6 | `remove(7)` | 2 | 2 | 7 | B | `[4, 4]` | `{4:{0,1}}` | `True` |
| 7 | `remove(4)` | 0 | 1 | 4 | C | `[4]` | `{4:{0}}` | `True` |
| 8 | `remove(4)` | 0 | 0 | 4 | B | `[]` | `{}` | `True` |
| 9 | `remove(4)` | | | | không có | `[]` | `{}` | `False` |

Chỉ có lệnh 9 trả `False` (trong các lệnh `remove`). Chuỗi này không có tình huống A vì sau lệnh 6 chỉ còn toàn số `4`.

Cell kế tiếp chạy lại đoạn code Bài 1 và mô phỏng Bài 2 (dùng `remove_with_choice` với `i = min(pos[val])`) để kiểm chứng; cell sau nữa là **đáp án Bài 3–6** kèm assert.

In [33]:
# Kiểm chứng Bài 1
a = [4, 8, 15, 16, 23]
i = 1
a[i] = a[-1]
a.pop()
print(a)
s = {3, 5}
s.add(5)
s.discard(9)
print(len(s), 9 in s)
pos = defaultdict(set)
print(len(pos), 7 in pos, pos[7], 7 in pos)
print(Counter([2, 2, 2, 6])[2] / 4)
print()

# Kiểm chứng Bài 2: luôn lấy chỉ số NHỎ NHẤT
nums, pos = [], defaultdict(set)
for op, val in [("insert", 4), ("insert", 4), ("insert", 7), ("insert", 4),
                ("remove", 4), ("remove", 7), ("remove", 4), ("remove", 4), ("remove", 4)]:
    if op == "insert":
        pos[val].add(len(nums))
        nums.append(val)
        res = len(pos[val]) == 1
    elif val in pos:
        remove_with_choice(nums, pos, val, min(pos[val]))
        res = True
    else:
        res = False
    print(f"{op}({val}) -> {res!s:<5} nums={nums} pos={fmt_pos(pos)}")

[4, 23, 15, 16]
2 False
0 False set() True
0.75

insert(4) -> True  nums=[4] pos={4:{0}}
insert(4) -> False nums=[4, 4] pos={4:{0,1}}
insert(7) -> True  nums=[4, 4, 7] pos={4:{0,1}, 7:{2}}
insert(4) -> False nums=[4, 4, 7, 4] pos={4:{0,1,3}, 7:{2}}
remove(4) -> True  nums=[4, 4, 7] pos={4:{0,1}, 7:{2}}
remove(7) -> True  nums=[4, 4] pos={4:{0,1}}
remove(4) -> True  nums=[4] pos={4:{0}}
remove(4) -> True  nums=[] pos={}
remove(4) -> False nums=[] pos={}


In [34]:
import bisect
import itertools


class AnswerRandomizedCollectionExercise(RandomizedCollectionExercise):
    def remove(self, val: int) -> bool:
        if val not in self.pos:
            return False
        i = self.pos[val].pop()
        last_i = len(self.nums) - 1
        last = self.nums[last_i]
        self.nums[i] = last
        self.pos[last].add(i)
        self.pos[last].discard(last_i)
        self.nums.pop()
        if not self.pos[val]:
            del self.pos[val]
        return True


class AnswerRandomizedSet:
    def __init__(self):
        self.nums = []
        self.idx = {}

    def insert(self, val: int) -> bool:
        if val in self.idx:
            return False
        self.idx[val] = len(self.nums)
        self.nums.append(val)
        return True

    def remove(self, val: int) -> bool:
        if val not in self.idx:
            return False
        i, last = self.idx[val], self.nums[-1]
        self.nums[i] = last
        self.idx[last] = i
        self.nums.pop()
        del self.idx[val]
        return True

    def getRandom(self) -> int:
        return random.choice(self.nums)


class AnswerRandomizedCollectionPlus(RandomizedCollection):
    def __init__(self):
        super().__init__()
        self.distinct = AnswerRandomizedSet()     # "bài 380" cho các giá trị khác nhau

    def insert(self, val: int) -> bool:
        is_new = super().insert(val)
        if is_new:
            self.distinct.insert(val)
        return is_new

    def remove(self, val: int) -> bool:
        ok = super().remove(val)
        if ok and val not in self.pos:            # val vừa hết bản sao
            self.distinct.remove(val)
        return ok

    def count(self, val: int) -> int:
        return len(self.pos[val]) if val in self.pos else 0

    def getRandomDistinct(self) -> int:
        return self.distinct.getRandom()


class AnswerWeightedPick:
    def __init__(self, w: List[int]):
        self.prefix = list(itertools.accumulate(w))      # [1, 3] -> [1, 4]

    def pickIndex(self) -> int:
        r = random.randrange(self.prefix[-1])             # vị trí ảo trong [0, tổng)
        return bisect.bisect_right(self.prefix, r)        # chỉ số nhỏ nhất có prefix[i] > r


# Bài 3
for t, (ops, args, expected) in enumerate(tests, 1):
    expect(AnswerRandomizedCollectionExercise, ops, args, expected, seed=t)
for t, (ops, args) in enumerate(cases[:100]):
    check_run(AnswerRandomizedCollectionExercise, ops, args, seed=t)

# Bài 4
random.seed(0)
out = run_commands(AnswerRandomizedSet, ops380, args380)
assert [o for o, op in zip(out, ops380) if op != "getRandom"] == [None, True, False, True, True, False]
assert out[4] in (1, 2) and out[7] == 2

# Bài 5
random.seed(5)
obj = AnswerRandomizedCollectionPlus()
for v in [1, 1, 2]:
    obj.insert(v)
assert obj.count(1) == 2 and obj.count(2) == 1 and obj.count(9) == 0
freq = Counter(obj.getRandomDistinct() for _ in range(60000))
assert abs(freq[1] / 60000 - 0.5) < 0.01 and abs(freq[2] / 60000 - 0.5) < 0.01
obj.remove(2)
assert obj.count(2) == 0 and all(obj.getRandomDistinct() == 1 for _ in range(100))
for _ in range(300):                               # đối chiếu ngẫu nhiên với Counter
    o, oracle = AnswerRandomizedCollectionPlus(), Counter()
    for _ in range(random.randint(1, 30)):
        v = random.randint(0, 3)
        if random.random() < 0.6:
            o.insert(v); oracle[v] += 1
        elif oracle[v] > 0:
            o.remove(v); oracle[v] -= 1
        oracle = +oracle
        assert all(o.count(x) == oracle[x] for x in range(4))
        assert sorted(o.distinct.nums) == sorted(oracle)
        if oracle:
            assert o.getRandomDistinct() in oracle

# Bài 6
random.seed(1)
wp = AnswerWeightedPick([1, 3])
freq = Counter(wp.pickIndex() for _ in range(60000))
assert abs(freq[0] / 60000 - 0.25) < 0.01 and abs(freq[1] / 60000 - 0.75) < 0.01
wp = AnswerWeightedPick([2, 5, 3])
freq = Counter(wp.pickIndex() for _ in range(60000))
for i, w in enumerate([2, 5, 3]):
    assert abs(freq[i] / 60000 - w / 10) < 0.01
print("Bài 6: w=[2,5,3] ->", {i: round(freq[i] / 60000, 3) for i in range(3)}, "(lý thuyết 0.2 / 0.5 / 0.3)")
print("Đáp án tham khảo chạy PASS toàn bộ test!")

Bài 6: w=[2,5,3] -> {0: 0.203, 1: 0.499, 2: 0.299} (lý thuyết 0.2 / 0.5 / 0.3)
Đáp án tham khảo chạy PASS toàn bộ test!


## 19. Template cuối cùng nên nhớ

**Cách chính** — `insert` / `remove` `O(1)` trung bình, `getRandom` `O(1)`, bộ nhớ `O(n)`:

```python
import random
from collections import defaultdict

class RandomizedCollection:
    def __init__(self):
        self.nums = []                     # mọi bản sao
        self.pos = defaultdict(set)        # giá trị -> tập chỉ số

    def insert(self, val: int) -> bool:
        self.pos[val].add(len(self.nums))
        self.nums.append(val)
        return len(self.pos[val]) == 1

    def remove(self, val: int) -> bool:
        if val not in self.pos:
            return False
        i = self.pos[val].pop()
        last_i = len(self.nums) - 1
        last = self.nums[last_i]
        self.nums[i] = last
        self.pos[last].add(i)              # add trước
        self.pos[last].discard(last_i)     # discard sau
        self.nums.pop()
        if not self.pos[val]:
            del self.pos[val]
        return True

    def getRandom(self) -> int:
        return random.choice(self.nums)
```

**Câu thần chú:** *"List giữ mọi bản sao để random; dict giá trị → set chỉ số để tìm; xóa = đổi chỗ với cuối rồi pop; thêm địa chỉ mới trước, gạch địa chỉ cũ sau; set rỗng thì xóa key."*

**Ba tình huống của `remove`:** A bình thường · B xóa chính ô cuối (`i == last_i`) · C giá trị trùng phần tử cuối (`last == val`). Thứ tự `add` → `discard` xử lý đúng cả ba.

## 20. Checklist tự kiểm tra trước khi submit

- Tên class và method đúng y như đề: `RandomizedCollection`, `insert`, `remove`, `getRandom`?
- `self.nums`, `self.pos` được tạo **trong `__init__`** (không ở mức class)?
- `insert` **luôn** thêm (kể cả khi đã có), và trả `True` chỉ khi trước đó chưa có (`len(self.pos[val]) == 1` sau khi thêm)?
- `remove` chỉ xóa **một** bản sao, trả `False` khi không có, **không** làm sập chương trình khi collection rỗng?
- Xóa bằng **đổi chỗ với phần tử cuối rồi `pop()`**, không dùng `del nums[i]` / `nums.remove(val)`?
- Thứ tự `self.pos[last].add(i)` **trước** `self.pos[last].discard(last_i)`?
- Dùng `discard` (không phải `remove`) cho `last_i`, và dọn key khi `pos[val]` rỗng?
- `getRandom` chọn trên `self.nums` (mọi bản sao) chứ không phải trên các giá trị khác nhau?
- Đã test: Example 1, `remove` trên rỗng, thêm–xóa hết–thêm lại, toàn số trùng, xóa phần tử cuối, giá trị biên `±2^31`, và **kiểm chứng phân phối** bằng `Counter`?